In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Playground S6E7 — Student Health Risk (FE + SHAP selection + XGBoost + RealMLP stack)

A stacked ensemble for the 3-class, balanced-accuracy **student health risk** problem
(`unhealthy` / `at-risk` / `fit`), heavily imbalanced toward `at-risk`.

**Pipeline:**
1. **Feature engineering** — ratio/interaction features, nonlinear transforms, a large family of
   *categorical variables derived from the numeric columns* (domain-threshold bins **and**
   train-fitted quantile bins), and EDA-driven missingness/subpopulation flags.
2. **SHAP-based feature selection** — a quick XGBoost, CatBoost, and LightGBM are each fit on the
   target-encoded feature matrix, TreeExplainer SHAP values are computed per model, normalized, and
   averaged; per-encoded-column importances are folded back onto their *source* feature, and the
   smallest feature set reaching 95% cumulative averaged |SHAP| is kept.
3. **Two base learners** → multinomial-LR meta-learner: **XGBoost + RealMLP**.
4. **Seed-bagged outer stack** — the whole 5-fold stacking pass runs once per seed in `SEEDS`
   (fold splits *and* every learner's own randomness vary), and OOF/test predictions are averaged
   across seeds before the meta-learner.
5. **Cross-fitted isotonic calibration** of each base learner's OOF probabilities, kept only if it
   improves meta-learner OOF balanced accuracy.
6. **Per-class decision-weight search** on the chosen blend's OOF probabilities — a
   coordinate-ascent grid search, adopted only if it beats no-reweighting by a margin.

**Why this learner pair (v9).** Runs v5–v8 showed CatBoost, LightGBM, and LR add nothing over
XGBoost in the meta-learner — their OOF scores and errors are nearly identical to XGB's, so the
stack paid 3 extra learners' runtime for zero LB movement. Meanwhile the strongest *public* single
models for this competition are **RealMLP**-based (LB 0.9509). So the stack is cut to the two
genuinely different mechanisms: tuned gradient-boosted trees (XGBoost) and the **RealMLP** below —
a from-scratch PyTorch port of the public notebook
[`nawfeelrahman1124444/ps-s6-ep6-realmlp-0-95090`](https://www.kaggle.com/code/nawfeelrahman1124444/ps-s6-ep6-realmlp-0-95090)
(itself built on `yekenot/ps-s6-e7-realmlp-pytorch`). **No `pytabkit`** — the port needs only torch,
which Kaggle ships with (the reason the original RealMLP learner was dropped back in the FT era).

> **Note on the previous version.** The from-scratch FT-Transformer is replaced by the RealMLP
> port, and CatBoost / LightGBM / LR are dropped as base learners. CatBoost and LightGBM still
> power the 3-model SHAP feature-selection consensus, which is unchanged.

## What carries over unchanged
Feature engineering, multiclass OOF target encoding, 3-model SHAP feature selection, HPO on a
stratified sample (XGBoost only now), balanced-accuracy optimization, per-fold leakage discipline,
and GPU auto-detection.

## EDA facts this notebook relies on
- **Heavy target imbalance** — predicting `at-risk` for everything scores ~86% raw accuracy but
  only ~33% *balanced* accuracy, so every learner is class-balanced (weights).
- **Missingness concentrates in the strongest predictors** — `stress_level` (~12%) and
  `sleep_duration` (~11%) top the missingness table (per vad13irt's public EDA, LB 0.95075), so
  FE carries explicit per-column missing flags + a row missing count (v8 showed these help only
  the linear/neural branch — trees already see raw NaNs natively).
- **No train/test distribution shift** — transforms fitted on train (quantile edges, scalers,
  vocabularies) apply safely to test.
- **Most class-separable numerics:** `sleep_duration`, `bmi`, `exercise_duration`, `step_count`.

## GPU support
XGBoost and the RealMLP use the GPU when one is available (**Settings → Accelerator → GPU**);
detection is via `torch.cuda.is_available()`. The CatBoost/LightGBM instances in the SHAP
feature-selection step follow the same device flags as before.


In [ ]:
!pip install -q optuna catboost shap

In [ ]:
import os, time, warnings, json
_NOTEBOOK_T0 = time.time()  # brackets true pipeline wall-time for the run-metrics summary at the end
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils import resample
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
import torch
import torch.nn as nn
import optuna, shap

warnings.filterwarnings("ignore"); optuna.logging.set_verbosity(optuna.logging.WARNING)
RNG, N_FOLDS, N_TRIALS = 42, 5, 25
SEEDS = [42]                      # single seed (Phase 2 new learners); RNG stays fixed for HPO/feature selection
TARGET = "health_condition"

# --- Phase 3: leakage-safe self-training (pseudo-labeling) --------------------
# Add high-confidence, class-balanced TEST rows to each fold's TRAINING data, with
# pseudo-labels generated by a fold-local xgb (Pass 1) trained ONLY on that fold's
# real rows. Encoders stay fit on real train, and Xv/OOF rows never receive pseudo
# rows -> OOF stays honest and comparable to prior runs. LB is the arbiter.
PSEUDO_LABEL     = False    # master toggle for the two-pass augmented refit (Phase3: regressed, kept off)
PSEUDO_CONF      = 0.90     # min max-softmax of the fold-local xgb to accept a test row
PSEUDO_PER_CLASS = 20000    # per-class cap -> class-balanced pseudo set (balanced-acc metric)
PSEUDO_LOG       = []       # per-fold (size, [n0,n1,n2]); summarized into RUN_METRICS_JSON

# --- GPU configuration -------------------------------------------------------
# Set the Kaggle notebook Accelerator to "GPU" (Settings > Accelerator) to use
# this. Detected once via torch; each library gets its GPU flags below.
USE_GPU        = torch.cuda.is_available()
TORCH_DEVICE   = "cuda" if USE_GPU else "cpu"
print(f"GPU available: {USE_GPU}"
      + (f"  ({torch.cuda.get_device_name(0)})" if USE_GPU else ""))

def xgb_device_kwargs():
    # XGBoost >= 2.0: device='cuda' + tree_method='hist' (gpu_hist is deprecated)
    return {"tree_method": "hist", "device": "cuda" if USE_GPU else "cpu"}

def cat_device_kwargs():
    # CatBoost: task_type='GPU', devices='0' when a GPU is available
    return {"task_type": "GPU", "devices": "0"} if USE_GPU else {"task_type": "CPU"}

def lgb_device_kwargs():
    # LightGBM: deliberately CPU-only. Kaggle's stock wheel isn't built with GPU
    # support, and requesting device_type="gpu" against a CPU-only build fails
    # hard rather than falling back -- CPU LightGBM is fast enough not to matter.
    return {"device_type": "cpu"}


## 1. Load data and encode the target

Column-type detection is deferred until **after** feature engineering, so the new derived categoricals are routed correctly.

In [ ]:
df      = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
print("Train:", df.shape, " Test:", df_test.shape)

classes    = sorted(df[TARGET].unique())
cls_to_int = {c: i for i, c in enumerate(classes)}
int_to_cls = {i: c for c, i in cls_to_int.items()}
N_CLASSES  = len(classes)
print(f"Classes ({N_CLASSES}): {classes}")

y        = df[TARGET].map(cls_to_int).values
X        = df.drop(columns=["id", TARGET]).reset_index(drop=True)
X_test   = df_test.drop(columns=["id"]).reset_index(drop=True)
test_ids = df_test["id"].values

# raw schema (used by the feature-engineering block below)
RAW_NUM = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
           "step_count", "exercise_duration", "water_intake"]
RAW_CAT = ["diet_type", "stress_level", "sleep_quality",
           "physical_activity_level", "smoking_alcohol", "gender"]
RAW_NUM = [c for c in RAW_NUM if c in X.columns]
RAW_CAT = [c for c in RAW_CAT if c in X.columns]
print("Raw numeric:", RAW_NUM)
print("Raw categorical:", RAW_CAT)
print("Class balance:")
for c in classes: print(f"  {c}: {(df[TARGET] == c).mean():.3f}")

## 2. Feature engineering

Everything here is **unsupervised** (no target used), so it's leakage-safe to compute on train and
apply identically to test — and because the EDA showed *no train/test distribution shift*, quantile
edges fitted on train transfer cleanly to test.

Three families of features, plus a deliberately large set of derived categoricals so that the SHAP
selection step below has something meaningful to prune.

**(a) Ratio & interaction features (numeric).** The raw columns are individually informative, but
*combinations* often carry the physiological signal — e.g. calories burned *per minute of exercise*
(intensity), or heart rate *relative to* step count (a sedentary-but-elevated-HR signature). These
give the trees ready-made interactions they'd otherwise have to discover through deep splits:
`exercise_intensity`, `hydration_ratio`, `activity_ratio`, `cal_per_step`, `cal_per_kg_bmi`,
`steps_per_hr`, `water_per_cal`, `ex_per_step`, `metabolic_load`, `sedentary_index`,
`cardio_stress`, `activity_balance`, plus two composite scores (`fitness_score`, `sleep_hr_product`,
`ex_water_product`).

**(b) Nonlinear transforms (numeric).** `bmi_sq` captures the U-shaped health risk at both BMI
extremes; `log_steps` / `log_cal` tame right-skew; `sleep_dev` and `hr_dev` encode *distance from a
healthy set-point* (8 h sleep, ~70 bpm resting HR) rather than the raw level — a monotone feature is
easier for the linear/neural learners than a bump-shaped one.

**(c) Categorical variables *from* numeric columns (the requested transformation).** Two ways:
- **Domain-threshold bins** using clinical cut-points: `bmi_category` (under/normal/over/obese),
  `sleep_category`, `heart_rate_category`, `step_category`, `water_category`, `exercise_category`,
  `calorie_category`. These let the tree/neural learners treat "clinically overweight" as a discrete
  state, and — crucially — they feed the **multiclass target encoder**, which turns each bin into
  per-class rates (a smooth, powerful signal the raw float can't express directly).
- **Quantile bins** (`*_qbin`, quintiles) for the most separable numerics identified in the EDA
  (`sleep_duration`, `bmi`, `exercise_duration`, `step_count`, …). Edges are fit on **train only**
  and reused on test, so they're leakage-safe.

Binning the same numeric several different ways is intentionally redundant — the SHAP step keeps the
cuts that matter and drops the rest.

**(f) EDA-driven additions** (from [vad13irt's EDA notebook](https://www.kaggle.com/code/vad13irt/ps-s6e7-eda-ensemble-lb-0-95075),
LB 0.95075): per-column **missingness flags** plus a row-wise `n_missing` count — that EDA shows the
most-missing columns (`stress_level` ~12%, `sleep_duration` ~11%) are exactly the strongest
predictors, so absence itself may be informative, and the linear/neural learners otherwise impute it
away silently; an `exercise_zero` flag for the discrete does-not-exercise spike at exactly 0; and a
`cal_low_mode` flag for the bimodal calorie distribution (~1800 vs ~2200 modes, valley ≈ 2000),
crossed with `physical_activity_level`.


In [ ]:
EPS = 1e-3
def _safe(a, b): return a / (b + EPS)

def engineer_features(d):
    """Unsupervised feature engineering. Safe to apply independently to train and test."""
    d = d.copy()
    sleep, hr, bmi = d["sleep_duration"], d["heart_rate"], d["bmi"]
    cal,   steps   = d["calorie_expenditure"], d["step_count"]
    ex,    water   = d["exercise_duration"], d["water_intake"]

    # (a) ratios / interactions
    d["exercise_intensity"] = _safe(cal, ex)        # kcal per exercise-minute
    d["hydration_ratio"]    = _safe(water, steps)   # water per step
    d["activity_ratio"]     = _safe(ex, sleep)      # exercise vs sleep
    d["cal_per_step"]       = _safe(cal, steps)
    d["cal_per_kg_bmi"]     = _safe(cal, bmi)
    d["steps_per_hr"]       = _safe(steps, hr)
    d["water_per_cal"]      = _safe(water, cal)
    d["ex_per_step"]        = _safe(ex, steps)
    d["sleep_hr_product"]   = sleep * hr
    d["ex_water_product"]   = ex * water
    d["metabolic_load"]     = _safe(cal, sleep)     # burn per sleep-hour
    d["sedentary_index"]    = _safe(hr, steps)      # elevated HR, low movement
    d["cardio_stress"]      = _safe(hr, sleep)
    d["activity_balance"]   = _safe(ex, ex + sleep)
    d["fitness_score"]      = steps / 1000.0 + ex - _safe(bmi, 1.0) + sleep

    # (b) nonlinear transforms
    d["bmi_sq"]    = bmi ** 2
    d["log_steps"] = np.log1p(steps.clip(lower=0))
    d["log_cal"]   = np.log1p(cal.clip(lower=0))
    d["sleep_dev"] = (sleep - 8.0).abs()            # distance from ideal 8h
    d["hr_dev"]    = (hr - 70.0).abs()              # distance from ~70 bpm resting

    # (c1) domain-threshold categoricals from numeric (clinical cut-points)
    d["bmi_category"] = pd.cut(bmi, [-np.inf, 18.5, 25, 30, np.inf],
                               labels=["underweight", "normal", "overweight", "obese"]).astype("object")
    d["sleep_category"] = pd.cut(sleep, [-np.inf, 5, 8, np.inf],
                                 labels=["short", "normal", "long"]).astype("object")
    d["heart_rate_category"] = pd.cut(hr, [-np.inf, 60, 100, np.inf],
                                      labels=["low", "normal", "high"]).astype("object")
    d["step_category"] = pd.cut(steps, [-np.inf, 5000, 10000, np.inf],
                                labels=["low", "medium", "high"]).astype("object")
    d["water_category"] = pd.cut(water, [-np.inf, 1.5, 3, np.inf],
                                 labels=["low", "medium", "high"]).astype("object")
    d["exercise_category"] = pd.cut(ex, [-np.inf, 15, 45, np.inf],
                                    labels=["light", "moderate", "intense"]).astype("object")
    d["calorie_category"] = pd.cut(cal, [-np.inf, 1800, 2500, np.inf],
                                   labels=["low", "medium", "high"]).astype("object")
    # (d) targeted additions (run-log SHAP showed signal concentrates in the strong
    #     raw features; these combine/transform those rather than add weak ratios)
    #  d1) deviation / interaction transforms of the strongest numerics
    d["bmi_dev"]          = (bmi - 22.0).abs()          # monotone distance from healthy-center BMI
    d["sleep_dev_sq"]     = d["sleep_dev"] ** 2         # sharpen the sleep U-shape
    d["sleep_bmi_dev"]    = d["sleep_dev"] * d["bmi_dev"]   # compound distance-from-healthy
    d["health_dev_score"] = d["sleep_dev"] + d["bmi_dev"] + d["hr_dev"]  # additive set-point risk
    d["sleep_bmi"]        = sleep * bmi                 # product of the two strongest numerics
    d["bmi_hr"]           = bmi * hr                    # body-composition x cardio
    #  d2) categorical crosses of the high-|SHAP| categoricals (target-encoded downstream)
    d["stress_x_sleepqual"] = d["stress_level"].astype(str) + "_" + d["sleep_quality"].astype(str)
    d["stress_x_activity"]  = d["stress_level"].astype(str) + "_" + d["physical_activity_level"].astype(str)
    d["activity_x_smoking"] = d["physical_activity_level"].astype(str) + "_" + d["smoking_alcohol"].astype(str)
    d["diet_x_activity"]    = d["diet_type"].astype(str) + "_" + d["physical_activity_level"].astype(str)
    d["stress_x_smoking"]   = d["stress_level"].astype(str) + "_" + d["smoking_alcohol"].astype(str)
    #  d3) three-way crosses of the strongest categoricals (higher-order interaction)
    d["stress_activity_sleepqual"] = (d["stress_level"].astype(str) + "_"
                                      + d["physical_activity_level"].astype(str) + "_"
                                      + d["sleep_quality"].astype(str))
    d["stress_activity_smoking"]   = (d["stress_level"].astype(str) + "_"
                                      + d["physical_activity_level"].astype(str) + "_"
                                      + d["smoking_alcohol"].astype(str))
    #  d4) strong categorical x clinical-bin crosses (bin comes from a strong numeric)
    d["stress_x_bmicat"]   = d["stress_level"].astype(str) + "_" + d["bmi_category"].astype(str)
    d["activity_x_bmicat"] = d["physical_activity_level"].astype(str) + "_" + d["bmi_category"].astype(str)
    d["stress_x_sleepcat"] = d["stress_level"].astype(str) + "_" + d["sleep_category"].astype(str)
    #  (f) EDA-driven additions (from vad13irt's public EDA notebook, LB 0.95075):
    #  f1) missingness indicators. The most-missing columns (stress_level ~12%,
    #      sleep_duration ~11%) are exactly the strongest predictors, so absence
    #      itself may carry signal. Trees see raw NaN natively, but LR and the
    #      FT-Transformer median-impute it away -- these flags keep that
    #      information alive for every learner. (Cat columns already expose a
    #      "nan" level downstream; their flags here are deliberately redundant.)
    raw_cols = [c for c in RAW_NUM + RAW_CAT if c in d.columns]
    for c in raw_cols:
        d[f"{c}_missing"] = d[c].isna().astype(int)
    d["n_missing"] = d[raw_cols].isna().sum(axis=1).astype(int)
    #  f2) "does not exercise" subpopulation: exercise_duration has a discrete
    #      spike at exactly 0 that the smooth bell shape hides
    d["exercise_zero"] = (ex == 0).astype(int)
    #  f3) calorie-expenditure bimodality: two modes (~1800 / ~2200) with a
    #      valley near ~2000. Flag the low mode and cross it with activity level
    #      (the EDA flags this mode as worth probing against activity features).
    d["cal_low_mode"] = (cal < 2000).astype(int)
    d["calmode_x_activity"] = (d["cal_low_mode"].astype(str) + "_"
                               + d["physical_activity_level"].astype(str))
    return d

# (c2) quantile-bin categoricals — edges fit on TRAIN only, reused on test (leakage-safe)
QBIN_COLS = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
             "step_count", "exercise_duration", "water_intake",
             "fitness_score", "metabolic_load", "exercise_intensity",
             "health_dev_score"]

def fit_quantile_edges(d, cols, q=5):
    edges = {}
    for c in cols:
        e = np.unique(np.nanquantile(d[c].astype(float), np.linspace(0, 1, q + 1)))
        e[0], e[-1] = -np.inf, np.inf            # open the tails for unseen test values
        edges[c] = e
    return edges

# (e) within-group numeric deviations: how far a person's numeric sits from the mean
#     of others at the same category level. Feature-only (no target) -> leakage-safe;
#     group means are fit on TRAIN and unseen test levels fall back to the global mean.
GROUP_STATS = [("bmi", "physical_activity_level"),
               ("sleep_duration", "stress_level"),
               ("heart_rate", "physical_activity_level"),
               ("calorie_expenditure", "physical_activity_level")]

def fit_group_stats(d, specs):
    stats = {}
    for num, cat in specs:
        stats[(num, cat)] = (d.groupby(cat)[num].mean(), float(d[num].mean()))
    return stats

def apply_group_stats(d, stats):
    d = d.copy()
    for (num, cat), (gmean, gglob) in stats.items():
        d[f"{num}_vs_{cat}"] = d[num] - d[cat].map(gmean).fillna(gglob)
    return d

def apply_quantile_bins(d, edges):
    d = d.copy()
    for c, e in edges.items():
        labs = [f"q{i}" for i in range(len(e) - 1)]
        d[f"{c}_qbin"] = pd.cut(d[c].astype(float), bins=e, labels=labs,
                                include_lowest=True).astype("object")
    return d

# apply FE
X      = engineer_features(X)
X_test = engineer_features(X_test)
gstats = fit_group_stats(X, GROUP_STATS)           # train-only group means (feature-only, leakage-safe)
X      = apply_group_stats(X, gstats)
X_test = apply_group_stats(X_test, gstats)
qedges = fit_quantile_edges(X, QBIN_COLS, q=5)     # train-only edges
X      = apply_quantile_bins(X, qedges)
X_test = apply_quantile_bins(X_test, qedges)
print(f"After FE:  train {X.shape},  test {X_test.shape}")
print(f"Added {X.shape[1] - len(RAW_NUM) - len(RAW_CAT)} engineered columns")

### 2.1 Detect column types on the engineered frame

Same auto-detector as before, run *after* FE so the derived bins are treated as categorical (and later target-encoded) while ratios/transforms stay numeric. Categorical NaNs become an explicit `"nan"` string via the cast — missingness is a level, not a hole.

In [ ]:
def _is_stringy(s):
    if isinstance(s.dtype, pd.CategoricalDtype):   return True
    if s.dtype == object:                          return True
    if s.dtype.kind in ("O", "U", "S", "T"):       return True
    return pd.api.types.is_string_dtype(s) and not pd.api.types.is_numeric_dtype(s)

def detect_column_types(frame):
    cat = [c for c in frame.columns
           if _is_stringy(frame[c]) or (frame[c].dtype.kind in "iu" and frame[c].nunique() <= 12)]
    num = [c for c in frame.columns if c not in cat]
    return cat, num

CAT_COLS, NUM_COLS = detect_column_types(X)
for c in CAT_COLS:
    X[c] = X[c].astype(str); X_test[c] = X_test[c].astype(str)

print(f"Categorical ({len(CAT_COLS)}): {CAT_COLS}")
print(f"Numeric ({len(NUM_COLS)}): {NUM_COLS}")

## 3. Multiclass OOF target encoding + balanced sample weights

Each categorical column expands into K encoded columns (per-class rate), OOF + smoothed to prevent leakage. `balanced_sample_weight` gives the tree/linear learners class balancing; RealMLP gets balancing via resampling instead (below).

In [ ]:
def oof_target_encode_multiclass(X_tr, y_tr, X_te, cat_cols, n_classes,
                                 n_splits=5, seed=RNG, m=20.0):
    X_tr_enc = X_tr.drop(columns=cat_cols).copy()
    X_te_enc = X_te.drop(columns=cat_cols).copy()
    priors = np.array([(y_tr == k).mean() for k in range(n_classes)])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for col in cat_cols:
        oof = np.tile(priors, (len(X_tr), 1)).astype(float)
        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            sub = pd.DataFrame({col: X_tr[col].iloc[tr_idx].values})
            for k in range(n_classes):
                sub[f"y{k}"] = (y_tr[tr_idx] == k).astype(float)
            agg = sub.groupby(col).agg(["sum", "count"])
            for k in range(n_classes):
                s = agg[(f"y{k}", "sum")]; c = agg[(f"y{k}", "count")]
                enc = (s + priors[k] * m) / (c + m)
                oof[val_idx, k] = X_tr[col].iloc[val_idx].map(enc).fillna(priors[k]).values
        for k in range(n_classes):
            X_tr_enc[f"{col}__te{k}"] = oof[:, k]

        sub = pd.DataFrame({col: X_tr[col].values})
        for k in range(n_classes):
            sub[f"y{k}"] = (y_tr == k).astype(float)
        agg = sub.groupby(col).agg(["sum", "count"])
        for k in range(n_classes):
            s = agg[(f"y{k}", "sum")]; c = agg[(f"y{k}", "count")]
            enc = (s + priors[k] * m) / (c + m)
            X_te_enc[f"{col}__te{k}"] = X_te[col].map(enc).fillna(priors[k]).values
    return X_tr_enc, X_te_enc

def balanced_sample_weight(y_arr):
    return compute_sample_weight(class_weight="balanced", y=y_arr)

## 4. SHAP-based feature selection — averaged across XGBoost, CatBoost, and LightGBM

Feature engineering above produced a lot of columns, many deliberately redundant. Rather than guess
which survive, we let three different tree models *tell us* via SHAP, and average their opinions
instead of trusting one model's view of what matters.

**Method.**
1. Take a stratified 75/25 split; **multiclass-target-encode** the categoricals (5-fold OOF on the
   75% slice, applied to the 25% slice) so every column is numeric and leakage-safe.
2. Fit three quick, class-balanced models on the same encoded matrix: XGBoost, CatBoost, LightGBM.
3. Compute **TreeExplainer SHAP values** on the held-out 25% for each model separately. For a 3-class
   model SHAP returns one attribution per class; we take **mean |SHAP| across both samples and
   classes** as a single importance per *encoded* column, per model.
4. Each categorical expands into `K` target-encoded columns (`col__te0..te{K-1}`); we **sum their
   importances back onto the source feature**, so a categorical is judged as a whole rather than
   penalized for being split across columns.
5. **Normalize each model's importance ranking to sum to 1**, then average the three — this prevents
   whichever model happens to produce larger raw SHAP magnitudes from dominating. Each model's own
   top-20 is printed separately below for observability, alongside the averaged ranking actually used.
6. Rank original features by the averaged importance and keep the smallest set reaching
   **`SHAP_CUM` (95%)** of total importance, with a floor of `MIN_FEATURES`.

**Why average three models.** A single model's SHAP ranking reflects that model's particular biases
(e.g. XGBoost's depth-first greedy splits vs. LightGBM's leaf-wise growth vs. CatBoost's ordered
boosting) — features useful to one tree family but not another were previously at risk of being
dropped for the whole pipeline, including for LR and the FT-Transformer, which have no say in a
single-model selection. Averaging three different tree families gives a more robust consensus.

> Note: SHAP importance measures *contribution to these particular models*, not ground-truth causal
> relevance. It's a pragmatic filter, not a causal claim.


In [ ]:
SHAP_CUM     = 0.95     # keep features up to this cumulative share of |SHAP|
MIN_FEATURES = 12       # never drop below this many original features

# 1) stratified split + OOF target encoding (leakage-safe)
X_sel_tr, X_sel_va, y_sel_tr, y_sel_va = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RNG)
X_sel_tr = X_sel_tr.reset_index(drop=True); X_sel_va = X_sel_va.reset_index(drop=True)
Xtr_e, Xva_e = oof_target_encode_multiclass(
    X_sel_tr, y_sel_tr, X_sel_va, CAT_COLS, N_CLASSES, n_splits=5)

# 2) three fast, comparably-sized, class-balanced models on the same encoded matrix
sel_xgb = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
    colsample_bytree=0.8, objective="multi:softprob", num_class=N_CLASSES,
    eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
    n_jobs=-1, verbosity=0)
sel_xgb.fit(Xtr_e, y_sel_tr, sample_weight=balanced_sample_weight(y_sel_tr))

sel_cat = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6, loss_function="MultiClass",
    auto_class_weights="Balanced", random_seed=RNG, **cat_device_kwargs(),
    verbose=0, allow_writing_files=False)
sel_cat.fit(Xtr_e, y_sel_tr)

sel_lgb = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
    subsample_freq=1, colsample_bytree=0.8, objective="multiclass",
    num_class=N_CLASSES, class_weight="balanced", random_state=RNG,
    **lgb_device_kwargs(), n_jobs=-1, verbosity=-1)
sel_lgb.fit(Xtr_e, y_sel_tr)

SEL_MODELS = {"xgb": sel_xgb, "cat": sel_cat, "lgb": sel_lgb}

# 3) TreeExplainer SHAP per model -> mean|SHAP| per encoded column
def _encoded_importance(model, Xva_e):
    sv = np.asarray(shap.TreeExplainer(model).shap_values(Xva_e))
    if sv.ndim == 3 and sv.shape[0] == N_CLASSES:      # (K, n, f)
        return np.abs(sv).mean(axis=(0, 1))
    elif sv.ndim == 3 and sv.shape[-1] == N_CLASSES:   # (n, f, K)
        return np.abs(sv).mean(axis=(0, 2))
    else:                                              # (n, f)
        return np.abs(sv).mean(axis=0)

# 4) fold target-encoded columns back onto their source categorical
import re
def _to_original(colname):
    m = re.match(r"^(.*)__te\d+$", colname)
    return m.group(1) if m else colname

enc_cols = list(Xva_e.columns)
per_model_imp = {}   # name -> Series of original-feature importance, normalized to sum 1
for name, model in SEL_MODELS.items():
    enc_imp = _encoded_importance(model, Xva_e)
    assert len(enc_imp) == len(enc_cols)
    orig_imp = {}
    for col, v in zip(enc_cols, enc_imp):
        o = _to_original(col)
        orig_imp[o] = orig_imp.get(o, 0.0) + float(v)
    ser = pd.Series(orig_imp).sort_values(ascending=False)
    per_model_imp[name] = ser / ser.sum()          # normalize so scales don't dominate the average
    print(f"\nTop 20 by |SHAP| -- {name.upper()}:")
    for fname, val in per_model_imp[name].head(20).items():
        print(f"  {fname:<26} {val:8.5f}")

# 5) average the three normalized rankings
imp_ser = pd.concat(per_model_imp.values(), axis=1).fillna(0.0).mean(axis=1).sort_values(ascending=False)
print(f"\nTop 20 by averaged (XGB+CatBoost+LightGBM) |SHAP|:")
for fname, val in imp_ser.head(20).items():
    print(f"  {fname:<26} {val:8.5f}")

# 6) cumulative-importance cutoff on the averaged ranking
cum = (imp_ser / imp_ser.sum()).cumsum()
k = max(int((cum < SHAP_CUM).sum()) + 1, MIN_FEATURES)
k = min(k, len(imp_ser))
selected_features = list(imp_ser.index[:k])

print(f"\nRanked {len(imp_ser)} original features; keeping top {k} "
      f"(cumulative avg |SHAP| = {cum.iloc[k-1]:.3f})")
dropped = [c for c in imp_ser.index if c not in selected_features]
print(f"Dropped {len(dropped)}: {dropped}")

# 7) restrict the whole pipeline to the selected features
X      = X[selected_features].reset_index(drop=True)
X_test = X_test[selected_features].reset_index(drop=True)
CAT_COLS = [c for c in CAT_COLS if c in selected_features]
NUM_COLS = [c for c in NUM_COLS if c in selected_features]
print(f"\nFinal feature set: {len(selected_features)}  "
      f"(CAT={len(CAT_COLS)}, NUM={len(NUM_COLS)})")


## 5. Optuna HPO on 10% sample — optimizing balanced accuracy

Runs on the **SHAP-selected** feature set. Only **XGBoost** is tuned now — the RealMLP uses the
reference notebook's published config verbatim (it's already competition-proven at LB 0.95090, and
per-fold HPO on a deep model inside the outer stack would be prohibitively slow).

Each study starts with **15 random trials** before Optuna's TPE sampler begins optimizing
(`n_startup_trials`), and scoring uses early stopping. The study's top trials are printed as a
small table afterward, to help narrow search ranges by hand in future runs.


In [ ]:
# --- HPO skip toggle (R1+): reuse the stored Optuna optimum instead of re-running
#     the ~10-min study each kernel. Params = v14 best (trial 23, 3-fold bal_acc
#     0.95063 on the 10% sample); stable across v9-v14. Set False to re-tune.
SKIP_HPO = True
XGB_BEST_PARAMS = dict(
    n_estimators=787, learning_rate=0.017312, max_depth=4,
    min_child_weight=2.22238, subsample=0.6979, colsample_bytree=0.806688,
    reg_alpha=4.042009e-04, reg_lambda=6.07e-04, gamma=1.797154e-08)

X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.10, stratify=y, random_state=RNG)
X_sample = X_sample.reset_index(drop=True); y_sample = np.asarray(y_sample)


def cv_score_xgb(p):
    skf, sc = StratifiedKFold(3, shuffle=True, random_state=RNG), []
    for tr, va in skf.split(X_sample, y_sample):
        Xt, Xv = X_sample.iloc[tr].reset_index(drop=True), X_sample.iloc[va].reset_index(drop=True)
        yt, yv = y_sample[tr], y_sample[va]
        Xt_e, Xv_e = oof_target_encode_multiclass(Xt, yt, Xv, CAT_COLS, N_CLASSES, n_splits=3)
        m = xgb.XGBClassifier(**p, objective="multi:softprob", num_class=N_CLASSES,
                              eval_metric="mlogloss", **xgb_device_kwargs(), random_state=RNG,
                              n_jobs=-1, early_stopping_rounds=50, verbosity=0)
        m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt), eval_set=[(Xv_e, yv)], verbose=False)
        sc.append(balanced_accuracy_score(yv, m.predict(Xv_e)))
    return float(np.mean(sc))


def obj_xgb(t):
    return cv_score_xgb({
        "n_estimators":     t.suggest_int("n_estimators", 200, 1000),
        "learning_rate":    t.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":        t.suggest_int("max_depth", 3, 10),
        "min_child_weight": t.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample":        t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":        t.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":       t.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "gamma":            t.suggest_float("gamma", 1e-8, 5.0, log=True)})


TRIAL_COLS = ["number", "value", "duration"]
studies = {}
if SKIP_HPO:
    best = {"xgb": dict(XGB_BEST_PARAMS)}
    print("HPO skipped -- stored best params:", best["xgb"])
else:
    for name, obj, n in [("xgb", obj_xgb, N_TRIALS)]:
        t0 = time.time()
        sampler = optuna.samplers.TPESampler(seed=RNG, n_startup_trials=15)  # random trials before TPE kicks in
        s = optuna.create_study(direction="maximize", sampler=sampler)
        s.optimize(obj, n_trials=n, show_progress_bar=False)
        studies[name] = s
        print(f"  {name}: bal_acc={s.best_value:.5f}  ({time.time()-t0:.1f}s)")
        tdf = s.trials_dataframe()
        param_cols = [c for c in tdf.columns if c.startswith("params_")]
        print(tdf[TRIAL_COLS + param_cols].sort_values("value", ascending=False).head(15).to_string(index=False))
    best = {k: v.best_params for k, v in studies.items()}


## 6. Base-learner fit functions (XGBoost + RealMLP)

Each returns `(val_proba, test_proba)`, both shape `(n, K)`, with `yv` passed explicitly. Every
`fit_*` takes a `seed` argument threaded into its own randomness, so the outer seed-bagging loop
produces genuinely different model inits per seed.

### RealMLP — the neural base learner (from-scratch PyTorch port)
**RealMLP** (Holzmüller et al., 2024, *Better by Default*) is an MLP with a bag of small but
compounding tricks that make it the strongest "default" tabular net. This implementation is ported
from the public notebook
[`nawfeelrahman1124444/ps-s6-ep6-realmlp-0-95090`](https://www.kaggle.com/code/nawfeelrahman1124444/ps-s6-ep6-realmlp-0-95090)
(LB **0.95090**, itself built on `yekenot/ps-s6-e7-realmlp-pytorch`) — architecture, schedules,
loss, and `CONFIG` kept exactly as published. The pieces:

- **PBLD numeric embeddings** — each numeric becomes `[x, PReLU(cos-periodic basis)]`, letting the
  net see both the raw value and a learned frequency decomposition.
- **NTPLinear ensemble layers** — every linear layer holds `n_ens = 8` independent weight tensors
  (`einsum` over an ensemble axis), so one training run fits **8 parallel MLPs** whose softmax
  outputs are averaged at eval — a built-in bag.
- **Categorical handling** — small-cardinality columns are one-hot; larger ones get per-ensemble
  embeddings.
- **EMA of weights** (`decay 0.997875`) for stable validation, a **flat_cos** LR schedule with
  per-parameter-group multipliers, **label smoothing cos-annealed to 0**, scheduled dropout, and a
  front **ScalingLayer** acting as soft feature selection.
- **Balanced class weights** in the smoothed CE loss (the competition is heavily imbalanced), and
  the best epoch is chosen on **validation balanced accuracy** — both consistent with the rest of
  this notebook.

**What was adapted to this pipeline** (data plumbing only, no architecture changes):
1. String categoricals → train-fold integer codes with a **reserved unknown slot** (the original
   mapped unseen values to `-1` and clipped them onto level 0 — ours keeps unknowns distinct,
   matching this notebook's convention everywhere else).
2. Numerics are **median-imputed on train-fold stats** (the PBLD embedding can't consume NaN).
3. Our **multiclass OOF target encoding** is appended as numeric features, mirroring the original's
   per-fold `TargetEncoder` columns fed alongside the raw cat codes.
4. The seed is threaded through np/torch per outer fold.


In [ ]:
# E2/R1 toggles: append the 39 per-value raw-column TE features (the lever behind
# FT-PLR's 0.95034 OOF -- see _rawte_frames in the FT-PLR cell) to the SHAP-narrowed
# inputs of the tree/MLP learners. rv_ prefix avoids colliding with the CAT_COLS TE
# columns (a raw categorical appears in both encodings).
TREE_RAWTE = True   # rawTE for ALL tree/GBDT learners (xgb, lgbte, catte, hgbte, et)
MLP_RAWTE  = True


def _tree_te_frames(Xt, yt, Xv, Xte, tr_idx, va_idx, seed):
    """SHAP-narrowed CAT_COLS multiclass TE + the 39 rv_-prefixed per-value rawTE
    columns, fit per outer fold (leak-free). Shared by every tree/GBDT learner
    (fit_xgb/lgbte/catte/hgbte/et) so they all consume an identical feature frame."""
    Xt_e, Xv_e  = oof_target_encode_multiclass(Xt, yt, Xv,  CAT_COLS, N_CLASSES, n_splits=5, seed=seed)
    _,    Xte_e = oof_target_encode_multiclass(Xt, yt, Xte, CAT_COLS, N_CLASSES, n_splits=5, seed=seed)
    if TREE_RAWTE and tr_idx is not None:
        Zt, Zv, Ze = _rawte_frames(tr_idx, yt, va_idx, seed)   # fit per fold, leak-free
        Xt_e  = pd.concat([Xt_e.reset_index(drop=True),  Zt.add_prefix("rv_")], axis=1)
        Xv_e  = pd.concat([Xv_e.reset_index(drop=True),  Zv.add_prefix("rv_")], axis=1)
        Xte_e = pd.concat([Xte_e.reset_index(drop=True), Ze.add_prefix("rv_")], axis=1)
    return Xt_e, Xv_e, Xte_e


def fit_xgb(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None,
            pseudo_idx=None, pseudo_lab=None):
    Xt_e, Xv_e, Xte_e = _tree_te_frames(Xt, yt, Xv, Xte, tr_idx, va_idx, seed)
    yt_fit = yt
    if pseudo_idx is not None and len(pseudo_idx):
        # append high-confidence TEST rows (encoded by the train-fit encoders) as extra
        # TRAIN rows. Xte_e stays whole for prediction; Xv_e/yv stay real -> leak-free.
        Xt_e   = pd.concat([Xt_e.reset_index(drop=True),
                            Xte_e.iloc[pseudo_idx].reset_index(drop=True)],
                           axis=0, ignore_index=True)
        yt_fit = np.concatenate([np.asarray(yt), np.asarray(pseudo_lab)])
    m = xgb.XGBClassifier(**best["xgb"], objective="multi:softprob", num_class=N_CLASSES,
                          eval_metric="mlogloss", **xgb_device_kwargs(), random_state=seed,
                          n_jobs=-1, early_stopping_rounds=50, verbosity=0)
    m.fit(Xt_e, yt_fit, sample_weight=balanced_sample_weight(yt_fit), eval_set=[(Xv_e, yv)], verbose=False)
    return m.predict_proba(Xv_e), m.predict_proba(Xte_e)


# ===========================================================================
# RealMLP (Holzmüller et al., 2024) — from-scratch PyTorch port, multiclass.
# Ported from nawfeelrahman1124444/ps-s6-ep6-realmlp-0-95090 (LB 0.95090),
# itself built on yekenot/ps-s6-e7-realmlp-pytorch. Architecture, schedules,
# loss, and CONFIG are kept exactly as published; only the data plumbing is
# adapted (see fit_mlp below). No pytabkit — torch only.
# ===========================================================================
import math
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight


class NumericalPreprocessor:
    def __init__(self, tfms):
        self._tfms = [t for t in tfms
                      if t in ("median_center", "robust_scale", "smooth_clip", "l2_normalize")]

    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X, 0.75, axis=0) - np.quantile(X, 0.25, axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5 * (X.max(axis=0)[zero_idx] - X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0 / (q_diff + 1e-30)
            self._iqr_factors[q_diff == 0.0] = 0.0
        return self

    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm == "median_center":
                X -= self._median[None, :]
            elif tfm == "robust_scale":
                X *= self._iqr_factors[None, :]
            elif tfm == "smooth_clip":
                X = X / np.sqrt(1 + (X / 3) ** 2)
            elif tfm == "l2_normalize":
                norms = np.linalg.norm(X, axis=1, keepdims=True)
                X /= np.where(norms == 0, 1.0, norms)
        return X


class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.cat_dims = cat_dims
        self.onehot_features = []
        self.embed_layers = nn.ModuleList()
        self._embed_feature_indices = []
        for i, dim in enumerate(cat_dims):
            if dim <= onehot_thresh:
                self.onehot_features.append(i)
            else:
                emb = nn.ModuleList([nn.Embedding(dim, embed_dim) for _ in range(n_ens)])
                self.embed_layers.append(emb)
                self._embed_feature_indices.append(i)

    def forward(self, x):
        batch_size, n_ens, _ = x.shape
        features = []
        if self.onehot_features:
            onehot_x    = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_oh    = sum(onehot_dims)
            encoded     = torch.zeros(batch_size, n_ens, total_oh, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx:idx+1].long()
                encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(encoded)
        for emb_list, feat_idx in zip(self.embed_layers, self._embed_feature_indices):
            feat_embs = []
            for model_idx in range(self.n_ens):
                indices = x[:, model_idx, feat_idx:feat_idx+1].long()
                feat_embs.append(emb_list[model_idx](indices))
            feat_combined = torch.cat(feat_embs, dim=1)
            features.append(feat_combined)
        return torch.cat(features, dim=2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias   = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum("bki,kio->bko", x, self.weight) / math.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4,
                 freq_scale=0.1, activation=nn.GELU):
        super().__init__()
        self.n_ens      = n_ens
        self.n_features = n_features
        self.out_dim    = out_dim
        self.w1  = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1  = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2  = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) / math.sqrt(hidden_dim))
        self.b2  = nn.Parameter(torch.zeros(n_ens, n_features, out_dim - 1))
        self.act = activation()
        nn.init.uniform_(self.b1, -math.pi, math.pi)

    def forward(self, x):
        periodic    = torch.cos(2 * math.pi * (x.unsqueeze(-1) * self.w1.unsqueeze(0) + self.b1.unsqueeze(0)))
        transformed = self.act(torch.einsum("bkfh,kfhd->bkfd", periodic, self.w2) + self.b2.unsqueeze(0))
        feat        = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return feat.flatten(start_dim=2)


class RealMLP(nn.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg):
        super().__init__()
        n_ens      = cfg["n_ens"]
        embed_dim  = cfg["embed_dim"]
        self.n_ens = n_ens
        self.cate = CategoricalFeatureLayer(
            n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim,
            onehot_thresh=cfg["onehot_thresh"])
        self.num_embed = PBLDEmbedding(
            n_ens=n_ens, n_features=n_numerical,
            hidden_dim=cfg["pbld_hidden_dim"], out_dim=cfg["pbld_out_dim"],
            freq_scale=cfg["pbld_freq_scale"], activation=cfg["pbld_activation"])
        num_emb_dim = n_numerical * cfg["pbld_out_dim"]
        cat_emb_dim = sum(c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total_dim   = num_emb_dim + cat_emb_dim
        act = cfg["activation"]
        layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens=n_ens, n_features=total_dim))
        self._dropout_modules = []
        in_dim = total_dim
        for i, out_dim_h in enumerate(cfg["hidden_dims"]):
            linear = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=out_dim_h)
            if i == 0:
                self.first_linear = linear
            drop = nn.Dropout(cfg["dropout"])
            self._dropout_modules.append(drop)
            layers += [linear, act(), drop]
            in_dim = out_dim_h
        self.hidden       = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens=n_ens, in_features=in_dim, out_features=output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        x     = self.hidden(torch.cat([x_num, x_cat], dim=2))
        return F.softmax(self.output_layer(x), dim=2)


def apply_schedule(init_value, progress, sched, flat_ratio=0.3):
    if sched == "constant":
        return init_value
    elif sched == "cos":
        return init_value * (math.cos(math.pi * progress) + 1) / 2
    elif sched == "flat_cos":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (math.cos(math.pi * t) + 1) / 2
    elif sched == "flat_anneal":
        if progress < flat_ratio:
            return init_value
        t = (progress - flat_ratio) / (1 - flat_ratio)
        return init_value * (1 - t)
    elif sched == "sqrt_cos":
        return init_value * math.sqrt((math.cos(math.pi * progress) + 1) / 2)
    elif sched == "expm4t":
        return init_value * math.exp(-4 * progress)
    else:
        raise ValueError(f"Unknown schedule: '{sched}'")


def get_parameter_groups(model, p):
    first_linear_weight_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if "num_embed" in name:
            pbld_p.append(param)
        elif "scale" in name:
            scale_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif "bias" in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    LR = p["lr"]
    WD = p["weight_decay"]
    return [
        {"params": scale_p,   "lr": LR * p["lr_scale_mult"],         "weight_decay": WD * p["wd_scale_mult"],         "group": "scale"},
        {"params": pbld_p,    "lr": LR * p["pbld_lr_factor"],        "weight_decay": WD,                              "group": "pbld"},
        {"params": first_w_p, "lr": LR * p["first_layer_lr_factor"], "weight_decay": WD * p["first_layer_wd_factor"], "group": "first_w"},
        {"params": other_w_p, "lr": LR,                              "weight_decay": WD,                              "group": "other_w"},
        {"params": bias_p,    "lr": LR * p["lr_bias_mult"],          "weight_decay": WD * p["wd_bias_mult"],          "group": "bias"},
    ]


def smooth_ce_loss(y_true, y_pred, ls=0.0, class_weights=None):
    n_classes = y_pred.size(1)
    y_smooth  = torch.full_like(y_pred, ls / n_classes)
    y_smooth.scatter_(1, y_true.unsqueeze(1), 1.0 - ls + ls / n_classes)
    per_sample_loss = -(y_smooth * torch.log(y_pred.clamp(1e-15, 1))).sum(dim=1)
    if class_weights is not None:
        sample_weights = class_weights[y_true]
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum()
    return per_sample_loss.mean()


# exact published config from the reference notebook (only verbosity lowered)
CONFIG = {
    "n_ens":         8,
    "embed_dim":     8,
    "onehot_thresh": 8,
    "hidden_dims":   [512, 512, 512],
    "dropout":       0.06,
    "p_drop_sched":  "expm4t",
    "activation":    nn.SiLU,
    "add_front_scale": True,
    "pbld_hidden_dim": 20,
    "pbld_out_dim":    5,
    "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU,
    "pbld_lr_factor":  0.093,
    "lr":               0.01,
    "mom":              0.9,
    "sq_mom":           0.98,
    "lr_sched":        "flat_cos",
    "flat_ratio":       0.3,
    "first_layer_lr_factor": 1.0,
    "first_layer_wd_factor": 0.1,
    "lr_scale_mult":    10.0,
    "lr_bias_mult":     0.1,
    "weight_decay":     0.013,
    "wd_scale_mult":    0.1,
    "wd_bias_mult":     0.5,
    "ema_decay":        0.997875,
    "grad_clip":        1.2,
    "ls_eps":       0.04,
    "ls_eps_sched": "cos",
    "tfms": ["median_center", "robust_scale"],
    "epochs":    3,
    "train_bs":  256,
    "eval_bs":   10240,
    "verbosity": 1,
    "use_early_stopping":                     False,
    "early_stopping_additive_patience":       10,
    "early_stopping_multiplicative_patience": 1,
    "device":       "cuda",
    "random_state": 63,
}


class RealMLP_TD_Classifier:
    def __init__(self, **kwargs):
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train, y_train, X_val, y_val, cat_col_names=None, X_test=None):
        p             = self.params
        dev           = torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        verbose       = p["verbosity"]
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        X_tr_num  = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat  = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr      = np.asarray(y_train)
        y_v       = np.asarray(y_val)

        self.preprocessor_ = NumericalPreprocessor(p["tfms"])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num  = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        if cat_col_names:
            all_cat = [X_tr_cat, X_val_cat]
            if X_test is not None:
                all_cat.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims = (np.concatenate(all_cat, axis=0).max(axis=0) + 1).tolist()
        else:
            cat_dims = []
        self.cat_dims_ = cat_dims

        if cat_dims:
            cat_max   = np.array(cat_dims) - 1
            X_tr_cat  = np.clip(X_tr_cat,  0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        classes       = np.unique(y_tr)
        self.classes_ = classes
        weights_np    = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weights = torch.as_tensor(weights_np, dtype=torch.float32, device=dev)
        n_classes     = len(classes)

        self.model_ = RealMLP(output_dim=n_classes, cat_dims=cat_dims,
                              n_numerical=X_tr_num.shape[1], cfg=p).to(dev)

        param_groups = get_parameter_groups(self.model_, p)
        for g in param_groups:
            g["lr_base"] = g["lr"]
        optimizer = torch.optim.AdamW(param_groups, betas=(p["mom"], p["sq_mom"]))

        Xtn = torch.as_tensor(X_tr_num,  dtype=torch.float32, device=dev)
        Xtc = torch.as_tensor(X_tr_cat,  dtype=torch.long,    device=dev)
        ytt = torch.as_tensor(y_tr,      dtype=torch.long,    device=dev)
        Xvn = torch.as_tensor(X_val_num, dtype=torch.float32, device=dev)
        Xvc = torch.as_tensor(X_val_cat, dtype=torch.long,    device=dev)

        n_ens       = p["n_ens"]
        train_bs    = p["train_bs"]
        eval_bs     = p["eval_bs"]
        epochs      = p["epochs"]
        lr_sched    = p["lr_sched"]
        flat_ratio  = p["flat_ratio"]
        ema_decay   = p["ema_decay"]
        total_steps = epochs * len(y_tr)
        train_order = np.arange(len(y_tr))

        best_score     = -np.inf
        best_epoch     = 0
        best_val_probs = None
        best_state     = None
        ema_state      = None
        if ema_decay > 0:
            ema_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}

        for epoch in range(epochs):
            self.model_.train()
            for start in range(0, len(y_tr), train_bs):
                progress  = (epoch * len(y_tr) + start) / total_steps
                idx_batch = train_order[start:start + train_bs]
                for g in optimizer.param_groups:
                    g["lr"] = apply_schedule(g["lr_base"], progress, lr_sched, flat_ratio)
                optimizer.zero_grad()
                y_pred   = self.model_(Xtn[idx_batch], Xtc[idx_batch])
                ls_val   = apply_schedule(p["ls_eps"],  progress, p["ls_eps_sched"],  flat_ratio)
                drop_val = apply_schedule(p["dropout"], progress, p["p_drop_sched"],  flat_ratio)
                for dm in self.model_._dropout_modules:
                    dm.p = drop_val
                loss = smooth_ce_loss(
                    ytt[idx_batch].repeat_interleave(n_ens),
                    y_pred.reshape(-1, n_classes),
                    ls=ls_val, class_weights=class_weights)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), p["grad_clip"])
                optimizer.step()
                if ema_state is not None:
                    with torch.no_grad():
                        for key, value in self.model_.state_dict().items():
                            if torch.is_floating_point(value):
                                ema_state[key].mul_(ema_decay).add_(value.detach(), alpha=1.0 - ema_decay)
                            else:
                                ema_state[key].copy_(value)
            np.random.shuffle(train_order)

            self.model_.eval()
            live_state = None
            if ema_state is not None:
                live_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
                self.model_.load_state_dict(ema_state, strict=True)
            with torch.no_grad():
                val_probs = np.concatenate([
                    self.model_(Xvn[s:s + eval_bs], Xvc[s:s + eval_bs])
                        .mean(dim=1).cpu().numpy()
                    for s in range(0, len(y_v), eval_bs)
                ], axis=0)
            epoch_score = balanced_accuracy_score(y_v, np.argmax(val_probs, axis=1))
            improved    = epoch_score > best_score
            if improved:
                best_score     = epoch_score
                best_epoch     = epoch + 1
                best_val_probs = val_probs.copy()
                state_src      = ema_state if ema_state is not None else self.model_.state_dict()
                best_state     = {k: v.detach().clone() for k, v in state_src.items()}
            if verbose >= 2:
                print(f"  epoch {epoch + 1}/{epochs}  score = {epoch_score:.5f}  "
                      f"best = {best_score:.5f}  ls = {ls_val:.4f}  drop = {drop_val:.4f}"
                      + (" *" if improved else ""))
            if p["use_early_stopping"]:
                patience = (best_epoch * p["early_stopping_multiplicative_patience"]
                            + p["early_stopping_additive_patience"])
                if (epoch + 1) > patience:
                    if verbose >= 1:
                        print(f"  Early stopping at epoch {epoch + 1} (best epoch {best_epoch})")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state, strict=True)
        self.best_score_     = best_score
        self.best_val_probs_ = best_val_probs
        self._dev            = dev
        if verbose >= 1:
            print(f"    -> RealMLP best val bal_acc: {best_score:.5f}  (epoch {best_epoch})")
        return self

    def predict_proba(self, X):
        eval_bs = self.params["eval_bs"]
        X_num   = self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        X_cat   = np.clip(X[self.cat_col_names_].values.astype(np.int64), 0, np.array(self.cat_dims_) - 1)
        Xn = torch.as_tensor(X_num, dtype=torch.float32, device=self._dev)
        Xc = torch.as_tensor(X_cat, dtype=torch.long,    device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([
                self.model_(Xn[s:s + eval_bs], Xc[s:s + eval_bs])
                    .mean(dim=1).cpu().numpy()
                for s in range(0, len(X_num), eval_bs)
            ], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


MLP_PARAM_OVERRIDES = {}   # per-run CONFIG tweaks without editing the port


def fit_mlp(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """RealMLP base learner. Feature layout mirrors the reference notebook:
    numerics (fold-median-imputed) + our multiclass OOF target encoding as
    extra numerics, plus the categoricals as integer codes for embedding/
    one-hot -- unseen levels get a reserved unknown slot. With MLP_RAWTE the
    39 per-value raw-column TE features join the numerics (rv_ prefix)."""
    np.random.seed(seed); torch.manual_seed(seed)
    cat_here = [c for c in CAT_COLS if c in Xt.columns]

    Xt_e, Xv_e  = oof_target_encode_multiclass(Xt, yt, Xv,  cat_here, N_CLASSES, n_splits=5, seed=seed)
    _,    Xte_e = oof_target_encode_multiclass(Xt, yt, Xte, cat_here, N_CLASSES, n_splits=5, seed=seed)
    med = Xt_e.median()

    cat_maps = {c: {v: i for i, v in enumerate(sorted(Xt[c].astype(str).unique()))}
                for c in cat_here}
    cat_names = [f"{c}__code" for c in cat_here]

    def assemble(enc_frame, raw_frame):
        out = enc_frame.reset_index(drop=True).fillna(med)
        for c in cat_here:
            unk = len(cat_maps[c])
            out[f"{c}__code"] = (raw_frame[c].astype(str).map(cat_maps[c])
                                 .fillna(unk).astype(np.int64).values)
        return out

    Ft, Fv, Fte = assemble(Xt_e, Xt), assemble(Xv_e, Xv), assemble(Xte_e, Xte)
    if MLP_RAWTE and tr_idx is not None:
        Zt, Zv, Ze = _rawte_frames(tr_idx, yt, va_idx, seed)   # fit per fold, leak-free
        Ft  = pd.concat([Ft,  Zt.add_prefix("rv_")], axis=1)
        Fv  = pd.concat([Fv,  Zv.add_prefix("rv_")], axis=1)
        Fte = pd.concat([Fte, Ze.add_prefix("rv_")], axis=1)

    m = RealMLP_TD_Classifier(**MLP_PARAM_OVERRIDES)
    m.fit(Ft, yt, Fv, yv, cat_col_names=cat_names, X_test=Fte)
    return m.best_val_probs_, m.predict_proba(Fte)


In [ ]:
# ===========================================================================
# FT-Transformer with PLR numerical embeddings -- third base learner (v11).
# Pure-PyTorch port of aribaymane61/ps-s6e7-ft-transformer-plr-embeddings-0-95059
# (single-model LB 0.95059). The real lever is per-value target encoding of all
# 13 RAW columns -- numerics included, because S6E7's numerics are secretly
# high-cardinality categoricals (the generator resampled discrete source values).
# Architecture: d=128, 2 blocks, 8 heads, PLR numeric tokens + categorical
# embeddings + a [CLS] readout. Unlike fit_xgb/fit_mlp this learner reads the RAW
# 13 columns (via tr_idx/va_idx into RAW_ALL), NOT the SHAP-narrowed feature set,
# so it keeps full numeric cardinality for the target encoder. The per-value TE
# reuses this notebook's own leak-free oof_target_encode_multiclass (each raw
# column, stringified, -> K per-class-rate columns = 39), so it needs no sklearn
# multiclass TargetEncoder and runs on any pinned image. Balanced CE loss, 16
# epochs, cosine schedule, best-val-BA state restore. torch only.
# ===========================================================================
from sklearn.preprocessing import QuantileTransformer

# raw columns aligned to the positional row order of X / y / X_test
FTPLR_NUM  = [c for c in RAW_NUM if c in df.columns]
FTPLR_CAT  = [c for c in RAW_CAT if c in df.columns]
FTPLR_ALLC = FTPLR_NUM + FTPLR_CAT
RAW_ALL    = df[FTPLR_ALLC].reset_index(drop=True)
RAW_TEST   = df_test[FTPLR_ALLC].reset_index(drop=True)

# categorical vocab = union(train, test); NaN -> its own "__nan__" level, so there
# are no unseen levels at inference (matches this notebook's reserved-unknown rule)
_ftplr_cat_maps, FTPLR_CAT_CARD = {}, []
for _c in FTPLR_CAT:
    _vals = sorted(set(RAW_ALL[_c].fillna("__nan__").astype(str))
                   | set(RAW_TEST[_c].fillna("__nan__").astype(str)))
    _ftplr_cat_maps[_c] = {v: i for i, v in enumerate(_vals)}
    FTPLR_CAT_CARD.append(len(_vals))

def _ftplr_cat_codes(frame):
    a = np.zeros((len(frame), len(FTPLR_CAT)), np.int64)
    for j, c in enumerate(FTPLR_CAT):
        a[:, j] = (frame[c].fillna("__nan__").astype(str)
                   .map(_ftplr_cat_maps[c]).fillna(0).astype(np.int64).values)
    return a

# exact-value string frames of all 13 raw columns, for per-value target encoding
_ftplr_str_all  = pd.DataFrame({c: RAW_ALL[c].fillna("__nan__").astype(str)  for c in FTPLR_ALLC})
_ftplr_str_test = pd.DataFrame({c: RAW_TEST[c].fillna("__nan__").astype(str) for c in FTPLR_ALLC})

def _rawte_frames(tr_idx, yt, va_idx, seed):
    """Per-value multiclass TE of all 13 raw columns (numerics stringified), fit
    leak-free within the train fold via oof_target_encode_multiclass. Returns
    (train, val, test) DataFrames of 13*K = 39 columns named {col}__te{k}, all
    RangeIndex-aligned. Shared by fit_ftplr/fit_node (via _ftplr_build_arrays)
    and, as of E2/R1, fit_xgb/fit_mlp."""
    Str_tr = _ftplr_str_all.iloc[tr_idx].reset_index(drop=True)
    Str_va = _ftplr_str_all.iloc[va_idx].reset_index(drop=True)
    Zt_df, Zv_df = oof_target_encode_multiclass(Str_tr, yt, Str_va,          FTPLR_ALLC, N_CLASSES, n_splits=5, seed=seed)
    _,     Ze_df = oof_target_encode_multiclass(Str_tr, yt, _ftplr_str_test, FTPLR_ALLC, N_CLASSES, n_splits=5, seed=seed)
    return Zt_df, Zv_df, Ze_df


FTPLR_CFG = dict(
    d_token=128, n_blocks=2, n_heads=8,
    attn_dropout=0.2, ffn_dropout=0.1, res_dropout=0.0,
    plr_k=24, plr_sigma=0.1,
    batch_size=4096, lr=1e-3, weight_decay=1e-5,
    max_epochs=24, patience=5, warmup_frac=0.10,   # R2: 16->24 epochs (longer cosine)
    n_bag=3,   # R2: nets per fold (different torch seeds), probabilities averaged
)
FTPLR_USE_AMP = USE_GPU


class _PLRTokenizer(nn.Module):
    """Numeric -> PLR: ReLU(Linear[sin,cos](2*pi*v*x)); categorical -> embedding + bias."""
    def __init__(self, n_num, cat_card, d, k, sigma):
        super().__init__()
        self.n_num, self.k = n_num, k
        self.freq  = nn.Parameter(torch.randn(n_num, k) * sigma)
        self.lin_w = nn.Parameter(torch.empty(n_num, 2 * k, d)); nn.init.normal_(self.lin_w, std=(2 * k) ** -0.5)
        self.lin_b = nn.Parameter(torch.zeros(n_num, d))
        self.n_cat = len(cat_card)
        if self.n_cat:
            offs = np.concatenate([[0], np.cumsum(cat_card)[:-1]]).astype(np.int64)
            self.register_buffer("cat_off", torch.tensor(offs))
            self.cat_emb = nn.Embedding(int(sum(cat_card)), d); nn.init.normal_(self.cat_emb.weight, std=d ** -0.5)
            self.cat_b = nn.Parameter(torch.empty(self.n_cat, d)); nn.init.normal_(self.cat_b, std=d ** -0.5)

    def forward(self, xn, xc):
        pre = 2 * math.pi * xn[:, :, None] * self.freq[None]
        per = torch.cat([torch.sin(pre), torch.cos(pre)], dim=-1)
        num = torch.einsum("bnk,nkd->bnd", per, self.lin_w) + self.lin_b[None]
        toks = [F.relu(num)]
        if self.n_cat:
            toks.append(self.cat_emb(xc + self.cat_off[None]) + self.cat_b[None])
        return torch.cat(toks, dim=1)


class _FTBlock(nn.Module):
    def __init__(self, d, heads, ad, fd, rd, dff):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=ad, batch_first=True)
        self.n2 = nn.LayerNorm(d); self.l1 = nn.Linear(d, dff * 2); self.l2 = nn.Linear(dff, d)
        self.fd = nn.Dropout(fd); self.rd = nn.Dropout(rd)

    def forward(self, x):
        r = self.n1(x); a, _ = self.attn(r, r, r, need_weights=False); x = x + self.rd(a)
        r = self.n2(x); u, v = self.l1(r).chunk(2, dim=-1)
        x = x + self.rd(self.l2(self.fd(u * F.relu(v))))   # ReGLU FFN
        return x


class _FTPLR(nn.Module):
    def __init__(self, n_num, cat_card, d=128, n_blocks=2, heads=8,
                 ad=0.2, fd=0.1, rd=0.0, k=24, sigma=0.1, d_out=3):
        super().__init__()
        self.tok = _PLRTokenizer(n_num, cat_card, d, k, sigma)
        self.cls = nn.Parameter(torch.empty(1, 1, d)); nn.init.normal_(self.cls, std=d ** -0.5)
        dff = int(d * 4 / 3)
        self.blocks = nn.ModuleList([_FTBlock(d, heads, ad, fd, rd, dff) for _ in range(n_blocks)])
        self.hn = nn.LayerNorm(d); self.head = nn.Linear(d, d_out)

    def forward(self, xn, xc):
        x = self.tok(xn, xc)
        x = torch.cat([self.cls.expand(x.shape[0], -1, -1), x], dim=1)   # prepend [CLS]
        for b in self.blocks:
            x = b(x)
        return self.head(F.relu(self.hn(x[:, 0])))


def _ftplr_build_arrays(tr_idx, yt, va_idx, seed):
    """Per-fold leak-free numerics: per-value multiclass target encoding of all 13
    raw columns (via the notebook's own OOF encoder -- cross-fitted within the train
    fold so a row never encodes itself) + median-imputed raw numerics, then a joint
    quantile-normal transform. Returns train/val/test numeric arrays."""
    Zt_df, Zv_df, Ze_df = _rawte_frames(tr_idx, yt, va_idx, seed)
    Zt, Zv, Ze = Zt_df.values, Zv_df.values, Ze_df.values   # each (n, 13*K) = 39 cols
    imp = SimpleImputer(strategy="median")
    Rt = imp.fit_transform(RAW_ALL.iloc[tr_idx][FTPLR_NUM])
    Rv = imp.transform(RAW_ALL.iloc[va_idx][FTPLR_NUM])
    Re = imp.transform(RAW_TEST[FTPLR_NUM])
    Nt = np.hstack([Rt, Zt]); Nv = np.hstack([Rv, Zv]); Ne = np.hstack([Re, Ze])
    qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                             subsample=200_000, random_state=seed)
    Nt = qt.fit_transform(Nt).astype(np.float32)
    Nv = qt.transform(Nv).astype(np.float32)
    Ne = qt.transform(Ne).astype(np.float32)
    return Nt, Nv, Ne


def _ftplr_predict(model, Xn, Xc, bs=8192):
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(Xn), bs):
            with torch.autocast("cuda", enabled=FTPLR_USE_AMP):
                o = model(Xn[i:i + bs], Xc[i:i + bs])
            outs.append(F.softmax(o.float(), dim=1).cpu().numpy())
    return np.concatenate(outs, 0)


def fit_ftplr(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None,
              pseudo_idx=None, pseudo_lab=None):
    """FT-Transformer + PLR base learner. Reads the RAW 13 columns via tr_idx/va_idx
    (Xt/Xv/Xte are the SHAP-narrowed frames the tree/MLP learners use and are ignored
    here). Returns (val_proba, test_proba), softmax probabilities of shape (n, K)."""
    np.random.seed(seed); torch.manual_seed(seed)
    Nt, Nv, Ne = _ftplr_build_arrays(tr_idx, yt, va_idx, seed)   # arrays built with REAL yt
    Ct, Cv, Ce = (_ftplr_cat_codes(RAW_ALL.iloc[tr_idx]),
                  _ftplr_cat_codes(RAW_ALL.iloc[va_idx]),
                  _ftplr_cat_codes(RAW_TEST))
    if pseudo_idx is not None and len(pseudo_idx):
        # add high-confidence TEST rows (already transformed by the train-fit QT/
        # imputer/TE above) as extra TRAIN rows. No encoder saw a pseudo label.
        # Reassigning yt threads the augmented labels through cw / yt_t / n below.
        Nt = np.vstack([Nt, Ne[pseudo_idx]])
        Ct = np.vstack([Ct, Ce[pseudo_idx]])
        yt = np.concatenate([np.asarray(yt), np.asarray(pseudo_lab)])

    dev = torch.device(TORCH_DEVICE)
    Xn_tr = torch.tensor(Nt, device=dev); Xc_tr = torch.tensor(Ct, device=dev)
    Xn_va = torch.tensor(Nv, device=dev); Xc_va = torch.tensor(Cv, device=dev)
    Xn_te = torch.tensor(Ne, device=dev); Xc_te = torch.tensor(Ce, device=dev)
    yt_t  = torch.tensor(yt, device=dev, dtype=torch.long)

    cnt = np.bincount(yt, minlength=N_CLASSES).astype(np.float64)
    cw  = torch.tensor(len(yt) / (N_CLASSES * cnt), device=dev, dtype=torch.float32)  # balanced

    # R2: seed-bag n_bag nets per fold (different torch init/shuffling, same fold
    # data and TE arrays) and average their probabilities -- targeted seed bagging
    # applied only to the strongest learner, where it is cheapest.
    n_bag = int(FTPLR_CFG.get("n_bag", 1))
    vprs, tprs, bag_bas = [], [], []
    for b in range(n_bag):
        torch.manual_seed(seed + 101 * b)
        model = _FTPLR(Nt.shape[1], FTPLR_CAT_CARD, d=FTPLR_CFG["d_token"], n_blocks=FTPLR_CFG["n_blocks"],
                       heads=FTPLR_CFG["n_heads"], ad=FTPLR_CFG["attn_dropout"], fd=FTPLR_CFG["ffn_dropout"],
                       rd=FTPLR_CFG["res_dropout"], k=FTPLR_CFG["plr_k"], sigma=FTPLR_CFG["plr_sigma"],
                       d_out=N_CLASSES).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=FTPLR_CFG["lr"], weight_decay=FTPLR_CFG["weight_decay"])
        bs = FTPLR_CFG["batch_size"]; n = len(yt); spe = math.ceil(n / bs)
        total = spe * FTPLR_CFG["max_epochs"]; warm = max(1, int(total * FTPLR_CFG["warmup_frac"]))

        def _lr_at(s):
            if s < warm:
                return s / warm
            p = (s - warm) / max(1, total - warm)
            return 0.5 * (1 + math.cos(math.pi * p))

        sched  = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)
        crit   = nn.CrossEntropyLoss(weight=cw)
        scaler = torch.cuda.amp.GradScaler(enabled=FTPLR_USE_AMP)

        best_ba, best_state, bad = -1.0, None, 0
        for ep in range(FTPLR_CFG["max_epochs"]):
            model.train(); perm = torch.randperm(n, device=dev)
            for i in range(0, n, bs):
                idx = perm[i:i + bs]; opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", enabled=FTPLR_USE_AMP):
                    loss = crit(model(Xn_tr[idx], Xc_tr[idx]), yt_t[idx])
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            ba = balanced_accuracy_score(yv, _ftplr_predict(model, Xn_va, Xc_va).argmax(1))
            if ba > best_ba + 1e-5:
                best_ba, bad = ba, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= FTPLR_CFG["patience"]:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        print(f"    -> FT-PLR bag {b + 1}/{n_bag} best val bal_acc: {best_ba:.5f}")
        bag_bas.append(best_ba)
        vprs.append(_ftplr_predict(model, Xn_va, Xc_va))
        tprs.append(_ftplr_predict(model, Xn_te, Xc_te))
        del model
        if USE_GPU:
            torch.cuda.empty_cache()

    vpr, tpr = np.mean(vprs, 0), np.mean(tprs, 0)
    if n_bag > 1:
        print(f"    -> FT-PLR bag-avg val bal_acc: {balanced_accuracy_score(yv, vpr.argmax(1)):.5f}"
              f"  (members: {[round(s, 5) for s in bag_bas]})")
    del Xn_tr, Xc_tr, Xn_va, Xc_va, Xn_te, Xc_te, yt_t
    if USE_GPU:
        torch.cuda.empty_cache()
    return vpr, tpr


In [ ]:
# ===========================================================================
# Novel diversity base learners (v13/E3): native-categorical CatBoost experts.
# These do NOT target-encode -- CatBoost consumes the SHAP-selected categoricals
# directly (ordered boosting on raw cats), a genuinely different mechanism from
# the TE'd XGB and the FT-PLR / RealMLP nets. Configs are lifted from the public
# "ShadowCat" notebook (lucifer19/student-health-signal-engine, Kaggle GPU).
# Both fit the standard (Xt, yt, Xv, Xte, yv, seed) signature (no raw-column
# threading needed) and return (val_proba, test_proba) in class order [0,1,2].
# ===========================================================================
def fit_catnat(Xt, yt, Xv, Xte, yv, seed=RNG):
    """Native-categorical CatBoost core (depth 8, Bayesian bootstrap, balanced)."""
    cat_here = [c for c in CAT_COLS if c in Xt.columns]
    m = CatBoostClassifier(
        iterations=1500, learning_rate=0.045, depth=8, l2_leaf_reg=6.0,
        random_strength=0.35, bootstrap_type="Bayesian", bagging_temperature=0.35,
        loss_function="MultiClass", eval_metric="MultiClass",
        auto_class_weights="Balanced", od_type="Iter", od_wait=130,
        random_seed=seed, allow_writing_files=False, verbose=False,
        **cat_device_kwargs())
    m.fit(Xt, yt, cat_features=cat_here, eval_set=(Xv, yv), use_best_model=True)
    return m.predict_proba(Xv), m.predict_proba(Xte)


def fit_minority(Xt, yt, Xv, Xte, yv, seed=RNG):
    """Two minority-vs-rest binary CatBoosts (fit-vs-rest, unhealthy-vs-rest),
    combined via odds into a 3-class posterior with at-risk (class 0) as the odds-1
    baseline -- a decision boundary no softmax learner in the stack produces."""
    cat_here = [c for c in CAT_COLS if c in Xt.columns]

    def _bin(c):
        m = CatBoostClassifier(
            iterations=1000, learning_rate=0.06, depth=7, l2_leaf_reg=7.0,
            random_strength=0.55 + 0.20 * c, bootstrap_type="Bayesian", bagging_temperature=0.5,
            loss_function="Logloss", eval_metric="Logloss",
            auto_class_weights="SqrtBalanced", od_type="Iter", od_wait=110,
            random_seed=seed + 20 * c, allow_writing_files=False, verbose=False,
            **cat_device_kwargs())
        m.fit(Xt, (yt == c).astype(int), cat_features=cat_here,
              eval_set=(Xv, (yv == c).astype(int)), use_best_model=True)
        return m.predict_proba(Xv)[:, 1], m.predict_proba(Xte)[:, 1]

    def _combine(pf, pu):
        of = np.clip(pf, 1e-5, 1 - 1e-5) / np.clip(1 - pf, 1e-5, None)
        ou = np.clip(pu, 1e-5, 1 - 1e-5) / np.clip(1 - pu, 1e-5, None)
        raw = np.column_stack([np.ones(len(pf)), of, ou])   # [at-risk, fit, unhealthy]
        return raw / raw.sum(1, keepdims=True)

    pf_v, pf_t = _bin(1)   # fit-vs-rest
    pu_v, pu_t = _bin(2)   # unhealthy-vs-rest
    return _combine(pf_v, pu_v), _combine(pf_t, pu_t)


In [ ]:
# ===========================================================================
# NODE -- Neural Oblivious Decision Ensembles (v14, novel architecture).
# A neural network with a DECISION-TREE inductive bias: a layer of differentiable
# oblivious decision trees. Each tree, at each of `depth` levels, softly selects a
# feature (sparsemax over features -> ~1 feature) and applies a soft threshold; the
# 2**depth soft leaves are formed by the outer product of the per-level decisions,
# and leaf responses are SUMMED across many trees (an additive ensemble, GBDT-like).
# This occupies the one region of model space our other learners don't: neither an
# MLP/attention net (RealMLP, FT-PLR) nor a hard-split GBDT (XGB/CatBoost) -- so it
# is the best bet for a strong-yet-decorrelated member to lift the blend. Reuses
# FT-PLR's numeric pipeline (7 raw + 39 per-value-TE, quantile-normalized); no cat
# embeddings needed (the TE already encodes the categoricals). torch only.
# Ref: Popov, Morozov, Babenko 2019 (NODE); sparsemax: Martins & Astudillo 2016.
# ===========================================================================
def _sparsemax(z, dim=0):
    """Sparse projection onto the simplex (sharp feature selection; ~1 feature)."""
    z_sorted, _ = torch.sort(z, descending=True, dim=dim)
    zc = z_sorted.cumsum(dim) - 1
    rng = torch.arange(1, z.shape[dim] + 1, device=z.device)
    k = rng.view([-1 if i == dim else 1 for i in range(z.dim())])
    support = (1 + k * z_sorted) > zc
    kmax = support.sum(dim, keepdim=True)
    tau = zc.gather(dim, kmax - 1) / kmax
    return torch.clamp(z - tau, min=0.0)


class _ODST(nn.Module):
    """One layer of oblivious differentiable split trees, summed over trees."""
    def __init__(self, in_features, n_trees, depth, out_dim):
        super().__init__()
        self.n_trees, self.depth = n_trees, depth
        self.fs   = nn.Parameter(torch.randn(in_features, n_trees, depth))   # feature-select logits
        self.b    = nn.Parameter(torch.randn(n_trees, depth))                # thresholds (spread)
        self.logt = nn.Parameter(torch.zeros(n_trees, depth))                # log temperature
        self.resp = nn.Parameter(torch.randn(n_trees, 2 ** depth, out_dim) * 0.1)  # leaf responses
        codes = torch.tensor([[(l >> d) & 1 for d in range(depth)] for l in range(2 ** depth)],
                             dtype=torch.float32)
        self.register_buffer("codes", codes)     # (L, depth) in {0,1}

    def forward(self, x):                          # x: (B, F)
        fw   = _sparsemax(self.fs, dim=0)          # (F, T, D) sparse selection
        feat = torch.einsum("bf,ftd->btd", x, fw)  # (B, T, D)
        temp = F.softplus(self.logt) + 1e-6
        pr   = torch.sigmoid((feat - self.b[None]) / temp[None])   # (B, T, D) go-right prob
        assign = torch.ones(x.shape[0], self.n_trees, 2 ** self.depth, device=x.device)
        for d in range(self.depth):
            prd = pr[:, :, d:d + 1]                 # (B, T, 1)
            cd  = self.codes[:, d][None, None]      # (1, 1, L)
            assign = assign * (prd * cd + (1 - prd) * (1 - cd))
        return torch.einsum("btl,tlo->bo", assign, self.resp)      # (B, out_dim), summed over trees


class _DenseODST(nn.Module):
    """NODE v2: stacked ODST layers with DenseNet-style connections. Each layer's
    summed-tree output (a rep_dim representation) is concatenated onto the running
    feature set that the next layer selects over -- so layer 2 can build splits on
    layer-1 tree responses, giving depth the single layer lacked (v14 NODE was one
    layer, OOF 0.9493). Every layer keeps the working mechanism intact: sparsemax
    feature selection (~1 feature/level) + responses SUMMED over trees. The final
    layer emits N_CLASSES logits; a linear skip from all intermediate reps is added
    so early layers still reach the output (residual/boosting-like)."""
    def __init__(self, in_features, n_layers, n_trees, depth, rep_dim, out_dim):
        super().__init__()
        self.layers = nn.ModuleList()
        d = in_features
        for _ in range(n_layers):
            self.layers.append(_ODST(d, n_trees, depth, rep_dim))
            d += rep_dim
        self.skip = nn.Linear(n_layers * rep_dim, out_dim)

    def forward(self, x):
        h, reps = x, []
        for layer in self.layers:
            o = layer(h)                       # (B, rep_dim) summed over trees
            reps.append(o)
            h = torch.cat([h, o], dim=1)       # dense concat
        return self.skip(torch.cat(reps, dim=1))


NODE_CFG = dict(n_layers=2, n_trees=256, depth=6, rep_dim=32,
                lr=2e-3, weight_decay=1e-5,
                batch_size=2048, max_epochs=45, patience=8, warmup_frac=0.10)


def _node_predict(model, Xn, bs=8192):
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(Xn), bs):
            outs.append(F.softmax(model(Xn[i:i + bs]).float(), dim=1).cpu().numpy())
    return np.concatenate(outs, 0)


def fit_node(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """NODE base learner. Reads the raw 13 columns via tr_idx/va_idx (Xt/Xv/Xte are
    the SHAP-narrowed frames the tree learners use and are ignored here). Input is the
    same 46 numerics as FT-PLR (raw + per-value TE, quantile-normalized). v2 uses a
    2-layer DenseODST. fp32 (no AMP, for the sparsemax sort). Returns (val_proba,
    test_proba), softmax, shape (n, K)."""
    np.random.seed(seed); torch.manual_seed(seed)
    Nt, Nv, Ne = _ftplr_build_arrays(tr_idx, yt, va_idx, seed)   # (n, 46) each
    dev = torch.device(TORCH_DEVICE)
    Xn_tr = torch.tensor(Nt, device=dev); Xn_va = torch.tensor(Nv, device=dev); Xn_te = torch.tensor(Ne, device=dev)
    yt_t  = torch.tensor(yt, device=dev, dtype=torch.long)

    cnt = np.bincount(yt, minlength=N_CLASSES).astype(np.float64)
    cw  = torch.tensor(len(yt) / (N_CLASSES * cnt), device=dev, dtype=torch.float32)  # balanced
    model = _DenseODST(Nt.shape[1], n_layers=NODE_CFG["n_layers"], n_trees=NODE_CFG["n_trees"],
                       depth=NODE_CFG["depth"], rep_dim=NODE_CFG["rep_dim"], out_dim=N_CLASSES).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=NODE_CFG["lr"], weight_decay=NODE_CFG["weight_decay"])
    bs = NODE_CFG["batch_size"]; n = len(yt); spe = math.ceil(n / bs)
    total = spe * NODE_CFG["max_epochs"]; warm = max(1, int(total * NODE_CFG["warmup_frac"]))

    def _lr_at(s):
        if s < warm:
            return s / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)
    crit  = nn.CrossEntropyLoss(weight=cw)

    best_ba, best_state, bad = -1.0, None, 0
    for ep in range(NODE_CFG["max_epochs"]):
        model.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]; opt.zero_grad(set_to_none=True)
            loss = crit(model(Xn_tr[idx]), yt_t[idx])
            loss.backward(); opt.step(); sched.step()
        ba = balanced_accuracy_score(yv, _node_predict(model, Xn_va).argmax(1))
        if ba > best_ba + 1e-5:
            best_ba, bad = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= NODE_CFG["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"    -> NODE best val bal_acc: {best_ba:.5f}")
    vpr = _node_predict(model, Xn_va); tpr = _node_predict(model, Xn_te)
    del model, Xn_tr, Xn_va, Xn_te, yt_t
    if USE_GPU:
        torch.cuda.empty_cache()
    return vpr, tpr


In [ ]:
# ===========================================================================
# TabM -- parameter-efficient deep-ensemble MLP (R3, novel architecture).
# One shared 3-layer MLP whose every hidden Linear is BatchEnsemble-augmented:
# member i computes y_i = ((x * r_i) @ W) * s_i + b_i, so K quasi-independent
# MLPs share one weight matrix W -- the ensemble effect of K nets at ~1/K the
# parameters. Random-SIGN init of r/s decorrelates members from step 0 (the
# paper's key trick; ones-init collapses members to near-identical nets). Heads
# are fully per-member. Loss: balanced CE averaged over members; prediction:
# mean of member softmaxes. Input = same 46 numerics as NODE/FT-PLR (7 raw +
# 39 per-value TE, quantile-normalized). AMP + cosine schedule. torch only.
# Ref: Gorishniy, Kotelnikov, Babenko 2024 -- "TabM: Advancing Tabular DL With
# Parameter-Efficient Ensembling" (arXiv:2410.24210).
# ===========================================================================
class _BELinear(nn.Module):
    """BatchEnsemble linear: shared W + per-member input/output sign-ish scales."""
    def __init__(self, k, d_in, d_out):
        super().__init__()
        self.w = nn.Parameter(torch.empty(d_in, d_out))
        nn.init.uniform_(self.w, -1 / math.sqrt(d_in), 1 / math.sqrt(d_in))  # w is (in, out)
        self.r = nn.Parameter(torch.where(torch.rand(k, d_in) < 0.5, -1.0, 1.0))
        self.s = nn.Parameter(torch.where(torch.rand(k, d_out) < 0.5, -1.0, 1.0))
        self.b = nn.Parameter(torch.zeros(k, d_out))

    def forward(self, x):                        # x: (B, K, d_in) -> (B, K, d_out)
        return torch.einsum("bki,io->bko", x * self.r[None], self.w) * self.s[None] + self.b[None]


class _KHeads(nn.Module):
    """Fully independent per-member output layer (K separate weight matrices)."""
    def __init__(self, k, d_in, d_out):
        super().__init__()
        self.w = nn.Parameter(torch.randn(k, d_in, d_out) / math.sqrt(d_in))
        self.b = nn.Parameter(torch.zeros(k, d_out))

    def forward(self, x):                        # (B, K, d_in) -> (B, K, d_out)
        return torch.einsum("bki,kio->bko", x, self.w) + self.b[None]


class _TabM(nn.Module):
    def __init__(self, d_in, k, hidden, dropout, d_out):
        super().__init__()
        self.k = k
        layers, d = [], d_in
        for h in hidden:
            layers += [_BELinear(k, d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.body = nn.Sequential(*layers)
        self.head = _KHeads(k, d, d_out)

    def forward(self, x):                        # (B, F) -> (B, K, C) member logits
        x = x[:, None, :].expand(-1, self.k, -1)
        return self.head(self.body(x))


TABM_CFG = dict(k=32, hidden=[512, 512, 512], dropout=0.15,
                lr=2e-3, weight_decay=3e-4, batch_size=2048,
                max_epochs=20, patience=4, warmup_frac=0.05)
TABM_USE_AMP = USE_GPU


def _tabm_predict(model, Xn, bs=8192):
    """Mean of member softmaxes."""
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(Xn), bs):
            with torch.autocast("cuda", enabled=TABM_USE_AMP):
                o = model(Xn[i:i + bs])
            outs.append(F.softmax(o.float(), dim=2).mean(dim=1).cpu().numpy())
    return np.concatenate(outs, 0)


def fit_tabm(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """TabM base learner. Reads the raw 13 columns via tr_idx/va_idx (Xt/Xv/Xte are
    the SHAP-narrowed frames and are ignored here); input is the same 46 numerics
    as NODE/FT-PLR. Returns (val_proba, test_proba), softmax mean, shape (n, K)."""
    np.random.seed(seed); torch.manual_seed(seed)
    Nt, Nv, Ne = _ftplr_build_arrays(tr_idx, yt, va_idx, seed)   # (n, 46) each
    dev = torch.device(TORCH_DEVICE)
    Xn_tr = torch.tensor(Nt, device=dev); Xn_va = torch.tensor(Nv, device=dev); Xn_te = torch.tensor(Ne, device=dev)
    yt_t  = torch.tensor(yt, device=dev, dtype=torch.long)

    cnt = np.bincount(yt, minlength=N_CLASSES).astype(np.float64)
    cw  = torch.tensor(len(yt) / (N_CLASSES * cnt), device=dev, dtype=torch.float32)  # balanced
    model = _TabM(Nt.shape[1], k=TABM_CFG["k"], hidden=TABM_CFG["hidden"],
                  dropout=TABM_CFG["dropout"], d_out=N_CLASSES).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=TABM_CFG["lr"], weight_decay=TABM_CFG["weight_decay"])
    bs = TABM_CFG["batch_size"]; n = len(yt); spe = math.ceil(n / bs)
    total = spe * TABM_CFG["max_epochs"]; warm = max(1, int(total * TABM_CFG["warmup_frac"]))

    def _lr_at(s):
        if s < warm:
            return s / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * p))

    sched  = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)
    crit   = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.cuda.amp.GradScaler(enabled=TABM_USE_AMP)
    K = TABM_CFG["k"]

    best_ba, best_state, bad = -1.0, None, 0
    for ep in range(TABM_CFG["max_epochs"]):
        model.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]; opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=TABM_USE_AMP):
                logits = model(Xn_tr[idx])                       # (b, K, C)
                loss = crit(logits.reshape(-1, N_CLASSES), yt_t[idx].repeat_interleave(K))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        ba = balanced_accuracy_score(yv, _tabm_predict(model, Xn_va).argmax(1))
        if ba > best_ba + 1e-5:
            best_ba, bad = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= TABM_CFG["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"    -> TabM best val bal_acc: {best_ba:.5f}")
    vpr = _tabm_predict(model, Xn_va); tpr = _tabm_predict(model, Xn_te)
    del model, Xn_tr, Xn_va, Xn_te, yt_t
    if USE_GPU:
        torch.cuda.empty_cache()
    return vpr, tpr


In [ ]:
# ===========================================================================
# rawTE GBDT / tree diversity (Phase 2). Alternative strong learners on the SAME
# TE feature frame as fit_xgb (_tree_te_frames: SHAP-narrowed CAT_COLS multiclass
# TE + the 39 rawTE columns), but with DIFFERENT algorithms -> decorrelation from
# XGB's level-wise histogram boosting, for the winning ftplr:xgb blend. LightGBM is
# leaf-wise + GOSS; CatBoost is ordered boosting on symmetric/oblivious trees
# (distinct from the native-cat fit_catnat); ExtraTrees is massively-randomized
# BAGGED trees (a true third axis vs every booster). All take the standard threaded
# signature and return (val_proba, test_proba) in class order [0,1,2].
# ===========================================================================
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier


def fit_lgbte(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """LightGBM (leaf-wise growth) on the shared TE frame. CPU per CLAUDE.md
    (Kaggle's stock wheel is CPU-only). Balanced sample weights + early stopping."""
    Xt_e, Xv_e, Xte_e = _tree_te_frames(Xt, yt, Xv, Xte, tr_idx, va_idx, seed)
    m = lgb.LGBMClassifier(objective="multiclass", num_class=N_CLASSES,
                           n_estimators=1500, learning_rate=0.02, num_leaves=63,
                           min_child_samples=60, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.7, reg_lambda=2.0, reg_alpha=1e-3,
                           random_state=seed, n_jobs=-1, verbosity=-1, **lgb_device_kwargs())
    m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt),
          eval_set=[(Xv_e, yv)], eval_metric="multi_logloss",
          callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(0)])
    return m.predict_proba(Xv_e), m.predict_proba(Xte_e)


def fit_catte(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """CatBoost (ordered boosting, symmetric/oblivious trees) on the shared TE frame
    -- distinct from the native-categorical fit_catnat. GPU. Balanced class weights,
    best-val-model restore."""
    Xt_e, Xv_e, Xte_e = _tree_te_frames(Xt, yt, Xv, Xte, tr_idx, va_idx, seed)
    m = CatBoostClassifier(iterations=2500, learning_rate=0.03, depth=7, l2_leaf_reg=5.0,
                           random_strength=0.5, bootstrap_type="Bayesian", bagging_temperature=0.4,
                           loss_function="MultiClass", eval_metric="MultiClass",
                           auto_class_weights="Balanced", od_type="Iter", od_wait=150,
                           random_seed=seed, allow_writing_files=False, verbose=False,
                           **cat_device_kwargs())
    m.fit(Xt_e, yt, eval_set=(Xv_e, yv), use_best_model=True)
    return m.predict_proba(Xv_e), m.predict_proba(Xte_e)


def fit_hgbte(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """sklearn HistGradientBoosting on the shared TE frame. CPU. Uses an internal
    validation split for early stopping; balanced via sample weights."""
    Xt_e, Xv_e, Xte_e = _tree_te_frames(Xt, yt, Xv, Xte, tr_idx, va_idx, seed)
    m = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.04, max_leaf_nodes=48,
                                       l2_regularization=2.0, min_samples_leaf=40,
                                       early_stopping=True, validation_fraction=0.12,
                                       n_iter_no_change=30, random_state=seed)
    m.fit(Xt_e, yt, sample_weight=balanced_sample_weight(yt))
    return m.predict_proba(Xv_e), m.predict_proba(Xte_e)


def fit_et(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """ExtraTrees (massively-randomized BAGGED trees) on the shared TE frame -- the
    true third decorrelation axis vs every booster/net. CPU, class-balanced."""
    Xt_e, Xv_e, Xte_e = _tree_te_frames(Xt, yt, Xv, Xte, tr_idx, va_idx, seed)
    imp = SimpleImputer(strategy="median")   # ExtraTrees can't consume NaN (the GBDTs can)
    Xt_i = imp.fit_transform(Xt_e); Xv_i = imp.transform(Xv_e); Xte_i = imp.transform(Xte_e)
    m = ExtraTreesClassifier(n_estimators=200, max_features=0.5, min_samples_leaf=50,
                             n_jobs=-1, class_weight="balanced", random_state=seed)  # cheap cfg: ~min not hours
    m.fit(Xt_i, yt)
    return m.predict_proba(Xv_i), m.predict_proba(Xte_i)


In [ ]:
# ===========================================================================
# Novel neural learners (Phase 2). Both consume FT-PLR's numeric pipeline
# (_ftplr_build_arrays -> 46 quantile-normed numerics = 7 raw + 39 per-value TE).
# fit_ftplr2 is a second FT-Transformer with a DIFFERENT shape (transformer-side
# diversity). fit_cnn1d is a 1D-CNN over the feature vector -- a convolutional
# inductive bias unlike any MLP/attention/tree learner in the stack. Both take the
# threaded signature and return softmax (val_proba, test_proba), shape (n, K).
# ===========================================================================
FTPLR2_CFG = dict(
    d_token=96, n_blocks=3, n_heads=6,
    attn_dropout=0.15, ffn_dropout=0.1, res_dropout=0.0,
    plr_k=32, plr_sigma=0.15,
    batch_size=4096, lr=1e-3, weight_decay=1e-5,
    max_epochs=20, patience=5, warmup_frac=0.10,
)


def fit_ftplr2(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """Second FT-Transformer (deeper/narrower than fit_ftplr) for transformer-side
    diversity. Single net (no bag). Reuses _FTPLR / _ftplr_build_arrays / _ftplr_cat_codes."""
    np.random.seed(seed); torch.manual_seed(seed + 7)
    cfg = FTPLR2_CFG
    Nt, Nv, Ne = _ftplr_build_arrays(tr_idx, yt, va_idx, seed)
    Ct, Cv, Ce = (_ftplr_cat_codes(RAW_ALL.iloc[tr_idx]),
                  _ftplr_cat_codes(RAW_ALL.iloc[va_idx]), _ftplr_cat_codes(RAW_TEST))
    dev = torch.device(TORCH_DEVICE)
    Xn_tr = torch.tensor(Nt, device=dev); Xc_tr = torch.tensor(Ct, device=dev)
    Xn_va = torch.tensor(Nv, device=dev); Xc_va = torch.tensor(Cv, device=dev)
    Xn_te = torch.tensor(Ne, device=dev); Xc_te = torch.tensor(Ce, device=dev)
    yt_t  = torch.tensor(yt, device=dev, dtype=torch.long)

    cnt = np.bincount(yt, minlength=N_CLASSES).astype(np.float64)
    cw  = torch.tensor(len(yt) / (N_CLASSES * cnt), device=dev, dtype=torch.float32)
    model = _FTPLR(Nt.shape[1], FTPLR_CAT_CARD, d=cfg["d_token"], n_blocks=cfg["n_blocks"],
                   heads=cfg["n_heads"], ad=cfg["attn_dropout"], fd=cfg["ffn_dropout"],
                   rd=cfg["res_dropout"], k=cfg["plr_k"], sigma=cfg["plr_sigma"],
                   d_out=N_CLASSES).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    bs = cfg["batch_size"]; n = len(yt); spe = math.ceil(n / bs)
    total = spe * cfg["max_epochs"]; warm = max(1, int(total * cfg["warmup_frac"]))
    def _lr_at(s):
        if s < warm: return s / warm
        p = (s - warm) / max(1, total - warm); return 0.5 * (1 + math.cos(math.pi * p))
    sched  = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)
    crit   = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.cuda.amp.GradScaler(enabled=FTPLR_USE_AMP)

    best_ba, best_state, bad = -1.0, None, 0
    for ep in range(cfg["max_epochs"]):
        model.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]; opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=FTPLR_USE_AMP):
                loss = crit(model(Xn_tr[idx], Xc_tr[idx]), yt_t[idx])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        ba = balanced_accuracy_score(yv, _ftplr_predict(model, Xn_va, Xc_va).argmax(1))
        if ba > best_ba + 1e-5:
            best_ba, bad = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]: break
    if best_state is not None: model.load_state_dict(best_state)
    print(f"    -> FT-PLR2 best val bal_acc: {best_ba:.5f}")
    vpr = _ftplr_predict(model, Xn_va, Xc_va); tpr = _ftplr_predict(model, Xn_te, Xc_te)
    del model, Xn_tr, Xc_tr, Xn_va, Xc_va, Xn_te, Xc_te, yt_t
    if USE_GPU: torch.cuda.empty_cache()
    return vpr, tpr


class _CNN1D(nn.Module):
    """1D-CNN over the 46-feature vector treated as a length-46 sequence (kernel=3
    convs give local structure; the 39 TE columns are grouped per source column so
    each column's 3 class-rates are adjacent). Global-avg-pool -> MLP head."""
    def __init__(self, n_feat, d_out, ch=64, k=3, p=0.1):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv1d(cin, cout, k, padding=k // 2),
                                 nn.BatchNorm1d(cout), nn.ReLU())
        self.net = nn.Sequential(block(1, ch), block(ch, ch), block(ch, ch * 2))
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                  nn.Linear(ch * 2, 128), nn.ReLU(), nn.Dropout(p),
                                  nn.Linear(128, d_out))

    def forward(self, x):                       # x: (B, F)
        return self.head(self.net(x[:, None, :]))


CNN1D_CFG = dict(ch=64, lr=2e-3, weight_decay=1e-4, batch_size=2048,
                 max_epochs=25, patience=5, warmup_frac=0.05)
CNN1D_USE_AMP = USE_GPU


def _cnn1d_predict(model, Xn, bs=8192):
    model.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(Xn), bs):
            with torch.autocast("cuda", enabled=CNN1D_USE_AMP):
                o = model(Xn[i:i + bs])
            outs.append(F.softmax(o.float(), dim=1).cpu().numpy())
    return np.concatenate(outs, 0)


def fit_cnn1d(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """1D-CNN base learner on the 46 FT-PLR numerics. torch, balanced CE, cosine."""
    np.random.seed(seed); torch.manual_seed(seed)
    Nt, Nv, Ne = _ftplr_build_arrays(tr_idx, yt, va_idx, seed)
    dev = torch.device(TORCH_DEVICE)
    Xn_tr = torch.tensor(Nt, device=dev); Xn_va = torch.tensor(Nv, device=dev); Xn_te = torch.tensor(Ne, device=dev)
    yt_t  = torch.tensor(yt, device=dev, dtype=torch.long)
    cnt = np.bincount(yt, minlength=N_CLASSES).astype(np.float64)
    cw  = torch.tensor(len(yt) / (N_CLASSES * cnt), device=dev, dtype=torch.float32)
    cfg = CNN1D_CFG
    model = _CNN1D(Nt.shape[1], N_CLASSES, ch=cfg["ch"]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    bs = cfg["batch_size"]; n = len(yt); spe = math.ceil(n / bs)
    total = spe * cfg["max_epochs"]; warm = max(1, int(total * cfg["warmup_frac"]))
    def _lr_at(s):
        if s < warm: return s / warm
        p = (s - warm) / max(1, total - warm); return 0.5 * (1 + math.cos(math.pi * p))
    sched  = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)
    crit   = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.cuda.amp.GradScaler(enabled=CNN1D_USE_AMP)

    best_ba, best_state, bad = -1.0, None, 0
    for ep in range(cfg["max_epochs"]):
        model.train(); perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]; opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=CNN1D_USE_AMP):
                loss = crit(model(Xn_tr[idx]), yt_t[idx])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        ba = balanced_accuracy_score(yv, _cnn1d_predict(model, Xn_va).argmax(1))
        if ba > best_ba + 1e-5:
            best_ba, bad = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["patience"]: break
    if best_state is not None: model.load_state_dict(best_state)
    print(f"    -> CNN1D best val bal_acc: {best_ba:.5f}")
    vpr = _cnn1d_predict(model, Xn_va); tpr = _cnn1d_predict(model, Xn_te)
    del model, Xn_tr, Xn_va, Xn_te, yt_t
    if USE_GPU: torch.cuda.empty_cache()
    return vpr, tpr

# ===========================================================================
# ModernNCA (Ye et al., 2024) -- retrieval/metric base learner (retuned). Learned
# MLP encoder -> BatchNorm'd embedding (magnitude preserved, unlike L2-norm);
# prediction is a soft nearest-neighbor vote over a class-balanced reference set
# drawn ONLY from the train fold (self-matches masked in training) -> leak-free,
# and a non-boosting/non-attention inductive bias for the decorrelated-axis probe.
# Distances are dim-normalized (cdist**2 / d_emb) so a learnable temperature is
# scale-stable. Class-balanced query+candidate sampling = the balanced-accuracy
# (uniform-prior Bayes) decision rule. v1 (L2-norm, small/short) hit only 0.690
# with at-risk recall 0.30 (mushy, conf 0.40) -> this config adds capacity, epochs,
# larger neighborhoods, magnitude, and consistent BN (query+candidate encoded jointly).
# ===========================================================================
NCA_CFG = dict(d_emb=256, hidden=512, cat_emb_dim=8, dropout=0.10,
               q_batch=1536, m_cand=6144, M_ref=24576, temp_init=0.20,
               lr=1e-3, weight_decay=1e-4, steps_per_epoch=600, max_epochs=40,
               patience=8, warmup_frac=0.10, n_bag=1)


class _NCAEncoder(nn.Module):
    """Per-categorical embedding + numerics -> MLP -> BatchNorm'd embedding. No per-
    sample L2-norm (that discards the rawTE magnitude signal); distances are
    dim-normalized by the caller instead."""
    def __init__(self, n_num, cat_card, d_emb=256, hidden=512, cat_emb_dim=8, dropout=0.1):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(int(c), cat_emb_dim) for c in cat_card])
        in_dim = n_num + len(cat_card) * cat_emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_emb))
        self.bn = nn.BatchNorm1d(d_emb)

    def forward(self, xn, xc):
        e = [emb(xc[:, j]) for j, emb in enumerate(self.embs)]
        h = torch.cat([xn] + e, dim=1) if e else xn
        return self.bn(self.mlp(h))


def _nca_predict(model, log_temp, Xn_ref, Xc_ref, y_ref, cls_idx, Xn_q, Xc_q, cfg, dev, eye, seed, bs=4096):
    """Soft-kNN vote over a FIXED class-balanced reference set sampled from the train
    fold (ref never includes the queries -> no self-match at inference)."""
    model.eval()
    rng = np.random.default_rng(seed)
    mpc = max(1, cfg["M_ref"] // len(cls_idx))
    ridx = np.concatenate([rng.choice(cls_idx[k], size=mpc, replace=len(cls_idx[k]) < mpc)
                           for k in range(len(cls_idx))])
    ridx_t = torch.tensor(ridx, device=dev)
    temp = log_temp.exp().clamp_min(1e-3)
    with torch.no_grad():
        zr = model(Xn_ref[ridx_t], Xc_ref[ridx_t]).float()          # (M, d)
        yr_oh = eye[y_ref[ridx_t]]                                  # (M, K)
        outs = []
        for i in range(0, len(Xn_q), bs):
            zq = model(Xn_q[i:i + bs], Xc_q[i:i + bs]).float()
            d = torch.cdist(zq, zr) ** 2 / zq.shape[1]
            outs.append((torch.softmax(-d / temp, dim=1) @ yr_oh).cpu().numpy())
    return np.concatenate(outs, 0)


def fit_nca(Xt, yt, Xv, Xte, yv, seed=RNG, tr_idx=None, va_idx=None):
    """ModernNCA base learner. Reads the RAW 13 cols via tr_idx/va_idx (Xt/Xv ignored).
    Neighbors (train + inference) drawn ONLY from the train fold; self-matches masked in
    training -> leak-free. Returns softmax (val_proba, test_proba), (n, K)."""
    np.random.seed(seed); torch.manual_seed(seed)
    cfg = NCA_CFG; ncl = N_CLASSES
    Nt, Nv, Ne = _ftplr_build_arrays(tr_idx, yt, va_idx, seed)
    Ct, Cv, Ce = (_ftplr_cat_codes(RAW_ALL.iloc[tr_idx]),
                  _ftplr_cat_codes(RAW_ALL.iloc[va_idx]), _ftplr_cat_codes(RAW_TEST))
    dev = torch.device(TORCH_DEVICE)
    Xn_tr = torch.tensor(Nt, device=dev); Xc_tr = torch.tensor(Ct, device=dev)
    Xn_va = torch.tensor(Nv, device=dev); Xc_va = torch.tensor(Cv, device=dev)
    Xn_te = torch.tensor(Ne, device=dev); Xc_te = torch.tensor(Ce, device=dev)
    yt_t  = torch.tensor(yt, device=dev, dtype=torch.long)
    cls_idx = [np.where(yt == k)[0] for k in range(ncl)]           # per-class train-fold pools
    eye = torch.eye(ncl, device=dev)

    model = _NCAEncoder(Nt.shape[1], FTPLR_CAT_CARD, d_emb=cfg["d_emb"], hidden=cfg["hidden"],
                        cat_emb_dim=cfg["cat_emb_dim"], dropout=cfg["dropout"]).to(dev)
    log_temp = nn.Parameter(torch.tensor(math.log(cfg["temp_init"]), device=dev))
    opt = torch.optim.AdamW(list(model.parameters()) + [log_temp],
                            lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    spe = cfg["steps_per_epoch"]; total = spe * cfg["max_epochs"]; warm = max(1, int(total * cfg["warmup_frac"]))
    def _lr_at(s):
        if s < warm: return s / warm
        p = (s - warm) / max(1, total - warm); return 0.5 * (1 + math.cos(math.pi * p))
    sched  = torch.optim.lr_scheduler.LambdaLR(opt, _lr_at)
    scaler = torch.cuda.amp.GradScaler(enabled=FTPLR_USE_AMP)

    qpc = max(1, cfg["q_batch"] // ncl); cpc = max(1, cfg["m_cand"] // ncl)
    def _sample_bal(per_class):
        return np.concatenate([np.random.choice(cls_idx[k], size=per_class,
                               replace=len(cls_idx[k]) < per_class) for k in range(ncl)])

    best_ba, best_state, best_temp, bad = -1.0, None, float(log_temp.detach()), 0
    for ep in range(cfg["max_epochs"]):
        model.train()
        for _ in range(spe):
            qi = torch.tensor(_sample_bal(qpc), device=dev); ci = torch.tensor(_sample_bal(cpc), device=dev)
            yq = yt_t[qi]; yc_oh = eye[yt_t[ci]]
            allid = torch.cat([qi, ci])
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=FTPLR_USE_AMP):
                z = model(Xn_tr[allid], Xc_tr[allid])                 # joint encode -> consistent BN
                zq = z[:len(qi)].float(); zc = z[len(qi):].float()
                d = torch.cdist(zq, zc) ** 2 / zq.shape[1]
                d = d.masked_fill(qi[:, None] == ci[None, :], float("inf"))   # no self-vote
                w = torch.softmax(-d / log_temp.exp().clamp_min(1e-3), dim=1)
                p = (w @ yc_oh).clamp_min(1e-9)
                loss = F.nll_loss(torch.log(p), yq)                   # class-balanced queries
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        vpr = _nca_predict(model, log_temp, Xn_tr, Xc_tr, yt_t, cls_idx, Xn_va, Xc_va, cfg, dev, eye, seed)
        ba = balanced_accuracy_score(yv, vpr.argmax(1))
        if ba > best_ba + 1e-5:
            best_ba, bad = ba, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            best_temp = float(log_temp.detach())
        else:
            bad += 1
            if bad >= cfg["patience"]: break
    if best_state is not None:
        model.load_state_dict(best_state)
        with torch.no_grad(): log_temp.copy_(torch.tensor(best_temp, device=dev))
    print(f"    -> ModernNCA best val bal_acc: {best_ba:.5f}")
    vpr = _nca_predict(model, log_temp, Xn_tr, Xc_tr, yt_t, cls_idx, Xn_va, Xc_va, cfg, dev, eye, seed)
    tpr = _nca_predict(model, log_temp, Xn_tr, Xc_tr, yt_t, cls_idx, Xn_te, Xc_te, cfg, dev, eye, seed)
    del model, Xn_tr, Xc_tr, Xn_va, Xc_va, Xn_te, Xc_te, yt_t
    if USE_GPU: torch.cuda.empty_cache()
    return vpr, tpr


## 7. Seed-bagged outer 5-fold stacking (2 base learners)

Meta feature matrix is `2 × K` wide; all predictions are out-of-fold. The entire 5-fold stacking
pass (`run_stack_for_seed`) runs once per seed in `SEEDS`, with every learner's own randomness (not
just the fold split) varying by seed; `oof_meta`/`test_meta` are averaged across seeds before
calibration and the meta-learner. **The RealMLP is the slow branch** — 3 epochs × an 8-member
internal ensemble per fold; on GPU each fold is minutes, on CPU budget accordingly.


In [ ]:
BASE_LEARNERS = ["nca"]            # Phase 4: ModernNCA (retrieval/metric novel architecture)
N_BASE = len(BASE_LEARNERS)
FIT_FN = {"xgb": fit_xgb, "mlp": fit_mlp, "ftplr": fit_ftplr, "catnat": fit_catnat,
          "minority": fit_minority, "node": fit_node, "tabm": fit_tabm,
          "lgbte": fit_lgbte, "catte": fit_catte, "hgbte": fit_hgbte, "et": fit_et,
          "ftplr2": fit_ftplr2, "cnn1d": fit_cnn1d, "nca": fit_nca}
def _slot(i): return slice(i * N_CLASSES, (i + 1) * N_CLASSES)

def _select_pseudo(test_proba, conf, per_class):
    """Class-balanced high-confidence pseudo-labels from a fold-local model's TEST
    proba (n,K): for each class take up to `per_class` most-confident rows whose
    max-softmax >= conf. Returns (idx int array, label int array)."""
    lab = test_proba.argmax(1); cval = test_proba.max(1)
    idxs, labs = [], []
    for k in range(N_CLASSES):
        cand = np.where((lab == k) & (cval >= conf))[0]
        if len(cand) > per_class:
            cand = cand[np.argsort(-cval[cand])[:per_class]]
        idxs.append(cand); labs.append(np.full(len(cand), k, dtype=int))
    return np.concatenate(idxs).astype(int), np.concatenate(labs)

def run_stack_for_seed(seed):
    """One full outer 5-fold stacking pass. Returns (oof_meta, test_meta, fold_scores)."""
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
    oof_meta_s  = np.zeros((len(X), N_BASE * N_CLASSES))
    test_meta_s = np.zeros((len(X_test), N_BASE * N_CLASSES))
    fold_scores_s = {k: [] for k in BASE_LEARNERS}

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        Xt = X.iloc[tr_idx].reset_index(drop=True); Xv = X.iloc[va_idx].reset_index(drop=True)
        yt, yv = y[tr_idx], y[va_idx]; t0 = time.time()

        # Phase 3 Pass 1: fold-local pseudo-labels from a cheap xgb trained ONLY on
        # this fold's real rows (keeps OOF leak-free), then class-balanced selection.
        pseudo_idx = pseudo_lab = None
        if PSEUDO_LABEL:
            _, _pt_gen = fit_xgb(Xt, yt, Xv, X_test, yv, seed=seed, tr_idx=tr_idx, va_idx=va_idx)
            pseudo_idx, pseudo_lab = _select_pseudo(_pt_gen, PSEUDO_CONF, PSEUDO_PER_CLASS)
            _dist = np.bincount(pseudo_lab, minlength=N_CLASSES).tolist()
            PSEUDO_LOG.append((int(len(pseudo_idx)), _dist))
            print(f"    fold {fold} pseudo: {len(pseudo_idx)} test rows  dist={_dist}  (conf>={PSEUDO_CONF})")

        # learners that consume the per-value rawTE features need the fold indices
        # into the raw frames (fit_ftplr/fit_node read ONLY those; fit_xgb/fit_mlp
        # append them to the SHAP-narrowed frames when their _RAWTE toggle is on).
        # Pass 2: xgb/ftplr refit on real + pseudo rows (threaded via pseudo_* kwargs).
        def _fit_one(k):
            extra = ({"tr_idx": tr_idx, "va_idx": va_idx}
                     if k in ("ftplr", "node", "xgb", "mlp", "tabm", "lgbte", "catte",
                              "hgbte", "et", "ftplr2", "cnn1d", "nca") else {})
            if PSEUDO_LABEL and k in ("xgb", "ftplr"):
                extra["pseudo_idx"], extra["pseudo_lab"] = pseudo_idx, pseudo_lab
            return FIT_FN[k](Xt, yt, Xv, X_test, yv, seed=seed, **extra)
        preds = [_fit_one(k) for k in BASE_LEARNERS]

        for i, (pv, pt) in enumerate(preds):
            oof_meta_s[va_idx, _slot(i)] = pv
            test_meta_s[:, _slot(i)]    += pt / N_FOLDS

        for i, k in enumerate(BASE_LEARNERS):
            fold_scores_s[k].append(balanced_accuracy_score(yv, oof_meta_s[va_idx, _slot(i)].argmax(1)))
        print(f"    fold {fold} ({time.time()-t0:.1f}s)  "
              + "  ".join(f"{k.upper()}={fold_scores_s[k][-1]:.5f}" for k in BASE_LEARNERS))
    return oof_meta_s, test_meta_s, fold_scores_s


oof_meta  = np.zeros((len(X), N_BASE * N_CLASSES))
test_meta = np.zeros((len(X_test), N_BASE * N_CLASSES))
seed_base_oof = {k: [] for k in BASE_LEARNERS}     # per-seed OOF balanced accuracy, for the run log

for seed in SEEDS:
    print(f"  Seed {seed}:")
    oof_s, test_s, fold_scores_s = run_stack_for_seed(seed)
    oof_meta  += oof_s  / len(SEEDS)
    test_meta += test_s / len(SEEDS)
    for i, k in enumerate(BASE_LEARNERS):
        seed_ba = balanced_accuracy_score(y, oof_s[:, _slot(i)].argmax(1))
        seed_base_oof[k].append(seed_ba)
        print(f"    {k:>7} OOF (seed {seed}): {seed_ba:.5f}")

print(f"\nBase-learner OOF balanced accuracy (averaged over {len(SEEDS)} seeds):")
base_oof_scores = {}
for i, k in enumerate(BASE_LEARNERS):
    ba = balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1))
    base_oof_scores[k] = ba
    print(f"  {k:>7}: {ba:.5f}  (per-seed: {[round(s, 5) for s in seed_base_oof[k]]})")


## 7.5 Cross-fitted calibration of base-learner OOF probabilities

`oof_meta`/`test_meta` hold each base learner's probabilities pre-calibration. Class-balancing
(sample weights, CatBoost's `auto_class_weights`) systematically distorts predicted probabilities in
exchange for better decision boundaries — stacking on distorted probabilities can make the
meta-learner's own `class_weight="balanced"` double-correct. We cross-fit an isotonic calibrator
(one-vs-rest per class, renormalized) per base learner: every OOF point is calibrated by a model that
never saw its label, so this stays leakage-safe by the same logic as the meta-learner's own OOF loop
below. Kept only if it improves meta-learner OOF balanced accuracy (validated, not assumed).


In [ ]:
def _calibrate_block(oof_block, y_true, test_block, n_splits=5, seed=RNG):
    """Cross-fit one-vs-rest isotonic calibration on a single base learner's K
    OOF probability columns. Returns (calibrated_oof, calibrated_test)."""
    n_classes = oof_block.shape[1]
    cal_oof = np.zeros_like(oof_block)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in kf.split(oof_block):
        for k in range(n_classes):
            ir = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
            ir.fit(oof_block[tr, k], (y_true[tr] == k).astype(float))
            cal_oof[va, k] = ir.predict(oof_block[va, k])
    row_sum = cal_oof.sum(axis=1, keepdims=True)
    cal_oof = np.divide(cal_oof, row_sum, out=np.full_like(cal_oof, 1.0 / n_classes), where=row_sum > 0)

    cal_test = np.zeros_like(test_block)
    for k in range(n_classes):
        ir = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        ir.fit(oof_block[:, k], (y_true == k).astype(float))
        cal_test[:, k] = ir.predict(test_block[:, k])
    row_sum = cal_test.sum(axis=1, keepdims=True)
    cal_test = np.divide(cal_test, row_sum, out=np.full_like(cal_test, 1.0 / n_classes), where=row_sum > 0)
    return cal_oof, cal_test


oof_meta_cal  = oof_meta.copy()
test_meta_cal = test_meta.copy()
for i, k in enumerate(BASE_LEARNERS):
    cal_oof, cal_test = _calibrate_block(oof_meta[:, _slot(i)], y, test_meta[:, _slot(i)])
    oof_meta_cal[:, _slot(i)]  = cal_oof
    test_meta_cal[:, _slot(i)] = cal_test
    print(f"  {k:>7} OOF bal_acc  raw={balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1)):.5f}"
          f"  calibrated={balanced_accuracy_score(y, oof_meta_cal[:, _slot(i)].argmax(1)):.5f}")


def _meta_oof_bal_acc(block):
    """Quick OOF balanced accuracy of the meta-learner trained on a given oof_meta variant."""
    proba = np.zeros((len(y), N_CLASSES))
    for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=RNG + 1).split(block, y):
        m = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, n_jobs=-1, random_state=RNG)
        m.fit(block[tr], y[tr])
        proba[va] = m.predict_proba(block[va])
    return balanced_accuracy_score(y, proba.argmax(1))

ba_raw = _meta_oof_bal_acc(oof_meta)
ba_cal = _meta_oof_bal_acc(oof_meta_cal)
CALIBRATE_BASE = ba_cal > ba_raw
print(f"\n  Meta OOF bal_acc  raw={ba_raw:.5f}  calibrated={ba_cal:.5f}  -> using calibration: {CALIBRATE_BASE}")

if CALIBRATE_BASE:
    oof_meta, test_meta = oof_meta_cal, test_meta_cal


## 8. Meta-learner + per-class decision-weight search

Trains on `oof_meta` (calibrated or not, per the toggle above). The meta-contribution readout (sum of
mean |coef| over each learner's class columns) tells you whether a given base learner is earning its
slot — a near-zero contribution means the meta-learner found it redundant with the trees, which is
fine, the stack ignores it gracefully.

The old binary prior-correction toggle (`meta_test_proba / priors`, used only if it beat the
uncorrected OOF score) is replaced by a general **coordinate-ascent grid search** over a per-class
decision-weight vector `w`: for a few rounds, each class's multiplier is scanned over a grid holding
the others fixed, keeping whichever value most improves OOF balanced accuracy. The result is compared
against `w = ones` (no correction) and `w = 1/priors` (the old heuristic), and whichever wins on OOF is
used for the final submission — this strictly generalizes the old toggle rather than replacing its
logic outright.


In [ ]:
# --- meta-learner: keep class_weight="balanced" (v5 showed dropping it collapses balanced
#     accuracy toward the majority class); sweep C. This is one blend candidate among several. ---
skf_meta = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RNG + 1)
def _fit_meta(C):
    """Full skf_meta OOF pass for one C. Returns (oof_proba, test_proba, mean|coef|)."""
    oof = np.zeros((len(X), N_CLASSES)); test = np.zeros((len(X_test), N_CLASSES)); coefs = []
    for tr, va in skf_meta.split(oof_meta, y):
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, n_jobs=-1, random_state=RNG)
        m.fit(oof_meta[tr], y[tr])
        oof[va] = m.predict_proba(oof_meta[va])
        test   += m.predict_proba(test_meta) / N_FOLDS
        coefs.append(np.abs(m.coef_).mean(axis=0))
    return oof, test, np.mean(coefs, axis=0)

meta_C_grid = [0.1, 0.3, 1.0, 3.0, 10.0]
meta_by_C = {}
for C in meta_C_grid:
    oof_c, test_c, mc_c = _fit_meta(C)
    ba_c = balanced_accuracy_score(y, oof_c.argmax(1))
    meta_by_C[C] = (ba_c, oof_c, test_c, mc_c)
    print(f"  meta C={C:<5}: OOF bal_acc={ba_c:.5f}")
meta_C = max(meta_by_C, key=lambda c: meta_by_C[c][0])
meta_bal_acc, meta_oof_proba, meta_test_proba, mc = meta_by_C[meta_C]
print(f"  -> best meta C={meta_C}  (OOF bal_acc={meta_bal_acc:.5f})")
print("  Meta contribution by base learner:")
for i, k in enumerate(BASE_LEARNERS):
    print(f"    {k:>7}: {mc[_slot(i)].sum():.3f}")

# --- blend candidates (v5 showed the 5-way simple average is dragged down by the weak LR;
#     compare a few clean blends and pick the best OOF -- few DOF, so low overfit risk) ---
def _avg(idxs, mat):
    return np.mean([mat[:, _slot(i)] for i in idxs], axis=0)

best_base_oof = max(base_oof_scores.values())
STRONG = [i for i, k in enumerate(BASE_LEARNERS) if base_oof_scores[k] >= best_base_oof - 0.01]
best_i = int(np.argmax([base_oof_scores[k] for k in BASE_LEARNERS]))
print(f"  strong learners (within 0.01 of best): {[BASE_LEARNERS[i] for i in STRONG]}"
      f"   best single: {BASE_LEARNERS[best_i]}")

avg_oof  = _avg(range(N_BASE), oof_meta)          # all 5 (kept as the simple_avg reference)
avg_test = _avg(range(N_BASE), test_meta)
avg_bal_acc = balanced_accuracy_score(y, avg_oof.argmax(1))

blend_candidates = {
    "avg_all":    (avg_oof, avg_test),
    "avg_strong": (_avg(STRONG, oof_meta), _avg(STRONG, test_meta)),
    "meta":       (meta_oof_proba, meta_test_proba),
    "best_base":  (oof_meta[:, _slot(best_i)], test_meta[:, _slot(best_i)]),
}
blend_scores = {name: balanced_accuracy_score(y, o.argmax(1)) for name, (o, t) in blend_candidates.items()}
blend_source = max(blend_scores, key=blend_scores.get)
blend_oof, blend_test = blend_candidates[blend_source]
blend_bal_acc = blend_scores[blend_source]
print("  Blend candidates (OOF): " + "  ".join(f"{n}={v:.5f}" for n, v in blend_scores.items()))
print(f"  -> blend: {blend_source}  ({blend_bal_acc:.5f})")

# --- per-class decision-weight search on the chosen blend, guarded against OOF overfit ---
priors = np.array([(y == k).mean() for k in range(N_CLASSES)])
WEIGHT_MARGIN = 5e-4   # only adopt reweighting if it beats 'none' by more than this on OOF

def optimize_class_weights(y_true, proba, n_rounds=3, grid=np.linspace(0.75, 1.35, 25)):
    w = np.ones(proba.shape[1])
    best_score = balanced_accuracy_score(y_true, proba.argmax(1))
    for _ in range(n_rounds):
        improved = False
        for k in range(proba.shape[1]):
            best_wk, best_local = w[k], best_score
            for cand in grid:
                w_try = w.copy(); w_try[k] = cand
                s = balanced_accuracy_score(y_true, (proba * w_try).argmax(1))
                if s > best_local:
                    best_local, best_wk = s, cand
            if best_local > best_score:
                w[k] = best_wk; best_score = best_local; improved = True
        if not improved:
            break
    return w, best_score

w_search, ba_search = optimize_class_weights(y, blend_oof)
w_prior = 1.0 / priors
ba_prior = balanced_accuracy_score(y, (blend_oof * w_prior).argmax(1))
ba_none  = blend_bal_acc

# 'none' is always eligible; reweighting must beat it by WEIGHT_MARGIN to be adopted
eligible = {"none": (np.ones(N_CLASSES), ba_none)}
if ba_prior  >= ba_none + WEIGHT_MARGIN: eligible["prior_inverse"] = (w_prior,  ba_prior)
if ba_search >= ba_none + WEIGHT_MARGIN: eligible["search"]        = (w_search, ba_search)
class_weight_method = max(eligible, key=lambda k: eligible[k][1])
class_weights, best_adj_bal_acc = eligible[class_weight_method]
print("  Class-weight candidates (OOF): "
      + f"none={ba_none:.5f}  prior_inverse={ba_prior:.5f}  search={ba_search:.5f}"
      + f"  (margin={WEIGHT_MARGIN})")
print(f"  -> using: {class_weight_method}  (bal_acc={best_adj_bal_acc:.5f}, "
      f"weights={np.round(class_weights, 3).tolist()})")

print("\n=== Summary ===")
for i, k in enumerate(BASE_LEARNERS):
    print(f"  {k:>7} OOF   : {balanced_accuracy_score(y, oof_meta[:, _slot(i)].argmax(1)):.5f}")
print(f"  simple avg  : {avg_bal_acc:.5f}")
print(f"  stacked meta: {meta_bal_acc:.5f}  (C={meta_C})")
print(f"  blend ({blend_source}) + class-weight ({class_weight_method}): {best_adj_bal_acc:.5f}")

## 9. Submission + prediction artifacts

Besides `submission.csv` (hard labels), the run saves the **probabilities** behind it —
`test_proba.csv` and `oof_proba.csv` (the latter with `y_true`) — so later runs can be blended
*across submissions* by `ply-s6e7-blend.ipynb`, with mixing weights re-optimized on the saved OOF.
This is the same mechanism the public LB-0.95075 notebook uses over public submission files.
`scripts/collect_run.py` archives these artifacts under `experiments/preds/<run_id>/`.

In [ ]:
final_pred = (blend_test * class_weights).argmax(1)
sub = pd.DataFrame({"id": test_ids, TARGET: [int_to_cls[i] for i in final_pred]})
sub.to_csv("submission.csv", index=False)
print("Submission shape:", sub.shape); print(sub.head())
print("\nPredicted class balance:")
print(sub[TARGET].value_counts(normalize=True))

run_metrics = {
    "n_seeds": len(SEEDS),
    "seeds": SEEDS,
    "hpo_sample_frac": 0.10,
    "notebook_runtime_sec": round(time.time() - _NOTEBOOK_T0, 1),
    "n_features_selected": len(selected_features),
    "base_oof_bal_acc": {k: round(float(base_oof_scores[k]), 5) for k in BASE_LEARNERS},
    "meta_oof_bal_acc_raw": round(float(meta_bal_acc), 5),
    "calibration_used": bool(CALIBRATE_BASE),
    "meta_oof_bal_acc_calibrated": round(float(ba_cal), 5),
    "meta_C": meta_C,
    "blend_source": blend_source,
    "simple_avg_oof_bal_acc": round(float(avg_bal_acc), 5),
    "class_weight_method": class_weight_method,
    "class_weights": np.round(class_weights, 4).tolist(),
    "final_oof_bal_acc": round(float(best_adj_bal_acc), 5),
    "predicted_class_balance": sub[TARGET].value_counts(normalize=True).round(4).to_dict(),
    "pseudo_label": ({"conf": PSEUDO_CONF, "per_class": PSEUDO_PER_CLASS, "per_fold": PSEUDO_LOG,
                      "avg_size": round(float(np.mean([s for s, _ in PSEUDO_LOG])), 1)}
                     if PSEUDO_LABEL and PSEUDO_LOG else None),
}
print("\nRUN_METRICS_JSON:" + json.dumps(run_metrics))

# --- prediction artifacts for cross-submission blending (ply-s6e7-blend.ipynb) ---
# Save the *probabilities* behind this submission, not just the argmax labels.
# blend_test/blend_oof are pre-decision-weight, so the blender can run its own
# per-class weight search after mixing runs (class_weights are in RUN_METRICS_JSON).
proba_cols = [int_to_cls[k] for k in range(N_CLASSES)]
test_art = pd.DataFrame({"id": test_ids})
oof_art  = pd.DataFrame({"id": df["id"].values,
                         "y_true": [int_to_cls[i] for i in y]})
for k, c in enumerate(proba_cols):
    test_art[c] = blend_test[:, k].round(6)
    oof_art[c]  = blend_oof[:, k].round(6)
test_art.to_csv("test_proba.csv", index=False)
oof_art.to_csv("oof_proba.csv", index=False)
print(f"\nSaved test_proba.csv {test_art.shape} and oof_proba.csv {oof_art.shape} "
      f"(blend source: {blend_source})")

# --- per-base-learner probability artifacts (for cross-submission blending / probe engine).
#     These are each learner's stacked OOF/test probs (post-calibration if it was adopted),
#     letting the blend/probe notebooks treat every learner as an independent source. ---
for i, k in enumerate(BASE_LEARNERS):
    lt = pd.DataFrame({"id": test_ids})
    lo = pd.DataFrame({"id": df["id"].values, "y_true": [int_to_cls[j] for j in y]})
    blk_t, blk_o = test_meta[:, _slot(i)], oof_meta[:, _slot(i)]
    for cidx, c in enumerate(proba_cols):
        lt[c] = blk_t[:, cidx].round(6)
        lo[c] = blk_o[:, cidx].round(6)
    lt.to_csv(f"test_proba_{k}.csv", index=False)
    lo.to_csv(f"oof_proba_{k}.csv", index=False)
    print(f"  saved test_proba_{k}.csv / oof_proba_{k}.csv")
